# VMamba-Small e Base congelados — K1, 672 px, RTX 4090

Derivado do avaliador Tiny que reproduziu exatamente a inspeção na 4090 e completou 100 queries. **Este notebook ainda não foi executado com Small/Base.**

1. Salve em `/workspace/ProtoSAM-repro/exp/03_vmamba_small_base_k1.ipynb`.
2. Encerre o kernel do Tiny para liberar a GPU. Abra este notebook com kernel `protosam`.
3. Execute com `VARIANTE="small"`, `MODO="inspecionar"`, `RETOMAR_DIR=None` (padrão). Faz uma query, salva máscaras, métricas, figura e confirma 42 chamadas CUDA (21 blocos × duas passagens ALP).
4. Após conferir a saída, reinicie o kernel, mude apenas `MODO="avaliar"`, salve e execute: 100 queries do Small.
5. Para Base: reinicie o kernel; use `VARIANTE="base"`, `MODO="inspecionar"`, `RETOMAR_DIR=None`; após a inspeção, reinicie e mude para avaliar.
6. Para retomar uma avaliação interrompida: mesma variante e modo, caminho explícito em `RETOMAR_DIR`, kernel reiniciado. A pasta deve ter assinatura idêntica de configuração, código, dados, pesos e ambiente.
7. Para consultar: `MODO="consultar"` e `CONSULTAR_DIR` explícito; não carrega modelos.

| Variante | Profundidades | Dimensão inicial | Canais finais | Drop path configurado |
|---|---|---:|---:|---:|
| Small | 2, 2, 15, 2 | 96 | 768 | 0,3 |
| Base | 2, 2, 15, 2 | 128 | 1024 | 0,6 |

Modelos em eval, sem treinamento. Grade nativa 21×21 → ALP 32×32. ALP oficial, val_wsize=2, SAM-H em canvas 1024, CCA=False, caixa + confiança + centroide, sem pontos negativos e sem cache do suporte. Normalização outnorm3 preservada com escala 1 e viés 0. O modo inspecionar é teste funcional, **não** compara máscaras Small/Base com Tiny como condição de aprovação e não escolhe parâmetros pelo teste.

Execução Tiny concluída (preservada): `/workspace/ProtoSAM-repro/reproducao/vmamba_tiny_k1_avaliar_20261008T095813008819Z`. Dice médio ALP 0,3492244582; SAM 0,4230823080; 100 queries, 65 melhoras e 35 pioras. Este notebook cria pastas Small/Base separadas. Não produz benchmark de tempo.


In [ ]:
from pathlib import Path
MODO = "inspecionar"  # consultar | inspecionar | avaliar
VARIANTE = "base"  # small | base
VARIANTES = {
    "small": {"depths": [2, 2, 15, 2], "dims": 96, "drop_path_rate": 0.3, "canais": 768},
    "base": {"depths": [2, 2, 15, 2], "dims": 128, "drop_path_rate": 0.6, "canais": 1024},
}
assert VARIANTE in VARIANTES
ARQUITETURA = VARIANTES[VARIANTE]
CANAIS_ENCODER = ARQUITETURA["canais"]
RAIZ = Path("/workspace/ProtoSAM-repro")
NOTEBOOK_ATUAL = RAIZ / "exp/03_vmamba_small_base_k1.ipynb"
REFERENCIA_K1 = RAIZ / "reproducao/01_avaliacao_K1_20261004T140725337013Z"

CONSULTAR_DIR = None  # preencha o caminho de uma execução Small/Base ao consultar
RETOMAR_DIR = None  # caminho explícito de uma execução criada por este consolidado
GPU_ESPERADA_AVALIACAO = "NVIDIA GeForce RTX 5090"
# Para uma avaliação exploratória deliberada na 4090, altere explicitamente acima.


In [ ]:
import os, sys, json, csv, hashlib, io, subprocess, random, platform
from datetime import datetime, timezone
import numpy as np
from PIL import Image
from IPython.display import display, Image as DisplayImage
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
assert MODO in ("consultar", "inspecionar", "avaliar")
assert RETOMAR_DIR is None or MODO != "consultar"

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with Path(caminho).open("rb") as f:
        for bloco in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def canonico(obj):
    return json.loads(json.dumps(obj, sort_keys=True))

def salvar_json(caminho, obj):
    caminho = Path(caminho)
    temporario = caminho.with_name(caminho.name + ".tmp")
    temporario.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(temporario, caminho)

def ler_json(caminho):
    return json.loads(Path(caminho).read_text(encoding="utf-8"))

def metricas_binarias(pred, gt):
    assert pred.shape == gt.shape == (1024, 1024)
    tp = int(np.count_nonzero(pred & gt))
    fp = int(np.count_nonzero(pred & ~gt))
    fn = int(np.count_nonzero(~pred & gt))
    # Nenhuma imagem é excluída. Convenção explícita para conjuntos vazios.
    return dict(TP=tp, FP=fp, FN=fn,
        Dice=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.0,
        IoU=tp/(tp+fp+fn) if tp+fp+fn else 1.0,
        precisao=tp/(tp+fp) if tp+fp else 0.0,
        recall=tp/(tp+fn) if tp+fn else 0.0)

def ler_mascara(caminho):
    with Image.open(caminho) as im: a = np.array(im)
    assert a.shape == (1024, 1024)
    assert np.isin(a, [0, 255]).all()
    return a > 0


## 1. Consulta sem inferência
A execução selecionada é explícita; não há seleção automática da pasta mais recente.

In [ ]:
if MODO == "consultar":
    assert CONSULTAR_DIR is not None, "Preencha CONSULTAR_DIR explicitamente"
    pasta = Path(CONSULTAR_DIR)
    if (pasta / "registro.json").is_file():
        historico = ler_json(pasta / "registro.json")
        print(json.dumps({k: historico[k] for k in ("tipo", "query", "metricas", "ambiente")}, indent=2, ensure_ascii=False))
        if (pasta / "comparacao.png").is_file():
            display(DisplayImage(filename=str(pasta / "comparacao.png")))
    else:
        print(json.dumps(ler_json(pasta / "status.json"), indent=2, ensure_ascii=False))
        print(json.dumps(ler_json(pasta / "resumo.json"), indent=2, ensure_ascii=False))
    print("Consulta concluída. Nenhuma inferência executada.")


## 2. Ambiente, origem do código e pesos
Células seguintes não carregam modelos no modo consultar. Código oficial comparado byte a byte com a base; `models/vmamba.py` é a adição auditada.

In [ ]:
if MODO != "consultar":
    from pathlib import Path
    import os
    import sys
    import hashlib
    import torch


    SHA_VMAMBA = "572207a2b6553ff85d81b602eebda2041623073f01fc0c6d1280c5a2add040f6"

    assert RAIZ.is_dir(), f"Repositório não encontrado: {RAIZ}"
    os.chdir(RAIZ)
    if str(RAIZ) not in sys.path:
        sys.path.insert(0, str(RAIZ))

    arquivo_vmamba = RAIZ / "models/vmamba.py"
    assert hashlib.sha256(arquivo_vmamba.read_bytes()).hexdigest() == SHA_VMAMBA

    from models import vmamba

    assert Path(vmamba.__file__).resolve() == arquivo_vmamba.resolve(), (
        f"Importação veio de outro local: {vmamba.__file__}"
    )

    CHECKPOINT_VMAMBA = RAIZ / f"pretrained_model/vmamba_{VARIANTE}_v2.pth"
    SUPORTE_K1 = "cju3xtufwiv9c0818djsc4cqd.png"
    RESOLUCAO_ENCODER = 672

    assert CHECKPOINT_VMAMBA.is_file()
    assert torch.cuda.is_available()

    print("Python:", sys.executable)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Código VMamba:", vmamba.__file__)
    print("SHA256 conferido:", SHA_VMAMBA)
    print("Checkpoint:", CHECKPOINT_VMAMBA.resolve())
    print("Suporte:", SUPORTE_K1)
    print("Resolução do encoder:", RESOLUCAO_ENCODER)
    import torch.nn.functional as F
    import inspect
    from models.grid_proto_fewshot import FewShotSeg
    from models.alpmodule import MultiProtoAsConv
    from models.ProtoSAM import ProtoSAM, ALPNetInput, ALPNetWrapper
    BASE_OFICIAL = "64868c909668aac9ad517f7192a2712e88268e8f"
    for relativo in ("models/grid_proto_fewshot.py", "models/alpmodule.py", "models/ProtoSAM.py",
                     "dataloaders/PolypDataset.py", "dataloaders/PolypTransforms.py",
                     "models/segment_anything/utils/transforms.py"):
        original = subprocess.run(["git", "-C", str(RAIZ), "show", f"{BASE_OFICIAL}:{relativo}"],
                                  capture_output=True, check=True).stdout
        assert (RAIZ / relativo).read_bytes() == original, f"Código oficial modificado: {relativo}"
    assert Path(inspect.getfile(FewShotSeg)).resolve() == RAIZ / "models/grid_proto_fewshot.py"
    assert Path(inspect.getfile(ProtoSAM)).resolve() == RAIZ / "models/ProtoSAM.py"
    if MODO == "avaliar":
        assert torch.cuda.get_device_name(0) == GPU_ESPERADA_AVALIACAO, "GPU diferente da configurada para avaliação"
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.set_num_threads(120)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.use_deterministic_algorithms(False)


In [ ]:
if MODO != "consultar":
    CONFIG_VMAMBA = dict(
        out_indices=(3,),
        pretrained=None,  # Carregaremos os pesos explicitamente abaixo.
        depths=ARQUITETURA["depths"],
        dims=ARQUITETURA["dims"],
        drop_path_rate=ARQUITETURA["drop_path_rate"],
        patch_size=4,
        in_chans=3,
        num_classes=1000,
        ssm_d_state=1,
        ssm_ratio=2.0,
        ssm_dt_rank="auto",
        ssm_act_layer="silu",
        ssm_conv=3,
        ssm_conv_bias=False,
        ssm_drop_rate=0.0,
        ssm_init="v0",
        forward_type="v05_noz",
        mlp_ratio=4.0,
        mlp_act_layer="gelu",
        mlp_drop_rate=0.0,
        gmlp=False,
        patch_norm=True,
        norm_layer="ln2d",
        downsample_version="v3",
        patchembed_version="v2",
        use_checkpoint=False,
        posembed=False,
        imgsize=224,
    )

    with torch.device("cpu"):
        encoder = vmamba.Backbone_VSSM(**CONFIG_VMAMBA)

    checkpoint = torch.load(
        CHECKPOINT_VMAMBA,
        map_location="cpu",
        weights_only=True,
    )
    pesos = checkpoint["model"]
    assert tuple(pesos["classifier.norm.weight"].shape) == (CANAIS_ENCODER,)
    blocos = {}
    for chave in pesos:
        partes = chave.split(".")
        if len(partes) > 3 and partes[0] == "layers" and partes[2] == "blocks":
            blocos.setdefault(int(partes[1]), set()).add(int(partes[3]))
    assert blocos == {i: set(range(d)) for i, d in enumerate(ARQUITETURA["depths"])}, blocos
    del pesos
    incompatibilidades = encoder.load_state_dict(
        checkpoint["model"].copy(),
        strict=False,
    )

    # Aceitamos somente as diferenças já identificadas na auditoria.
    ausentes_esperadas = {"outnorm3.weight", "outnorm3.bias"}
    extras_esperadas = {
        "classifier.norm.weight",
        "classifier.norm.bias",
        "classifier.head.weight",
        "classifier.head.bias",
    }

    assert set(incompatibilidades.missing_keys) == ausentes_esperadas, (
        f"Chaves ausentes inesperadas: {incompatibilidades.missing_keys}"
    )
    assert set(incompatibilidades.unexpected_keys) == extras_esperadas, (
        f"Chaves extras inesperadas: {incompatibilidades.unexpected_keys}"
    )

    # Preserva a normalização de saída usada na integração histórica.
    assert torch.equal(
        encoder.outnorm3.weight,
        torch.ones_like(encoder.outnorm3.weight),
    )
    assert torch.equal(
        encoder.outnorm3.bias,
        torch.zeros_like(encoder.outnorm3.bias),
    )

    encoder.eval()
    encoder.requires_grad_(False)

    assert all(not modulo.training for modulo in encoder.modules())
    assert all(not parametro.requires_grad for parametro in encoder.parameters())

    print("Pesos carregados com as incompatibilidades previstas.")
    print("Ausentes:", incompatibilidades.missing_keys)
    print("Descartadas:", incompatibilidades.unexpected_keys)
    print("Normalização de saída: escala 1 e viés 0.")
    print("Encoder em eval() e todos os parâmetros congelados.")
    print("Parâmetros:", f"{sum(p.numel() for p in encoder.parameters()):,}")

    del checkpoint

## 3. Adaptador VMamba → ALP
Somente extração de características, canais e grade são adaptados. O fallback de fundo da branch antiga não foi incorporado.

In [ ]:
if MODO != "consultar":
    from copy import deepcopy
    from util.consts import DEFAULT_FEATURE_SIZE

    CONFIG_ALP = {
        "which_model": f"vmamba_{VARIANTE}",
        "cls_name": "grid_proto",
        "proto_grid_size": 8,
        "lora": 0,
        "align": False,
        "debug": False,
        "use_coco_init": False,
        "use_pos_enc": False,
        "use_slice_adapter": False,
        "adapter_layers": 3,
        "dinov2_loss": False,
    }

    def construir_alp_vmamba(encoder_validado, resolucao, config):
        # A classe usa o encoder recebido, sem recarregar checkpoints.
        class FewShotSegVMamba(FewShotSeg):
            def get_encoder(self):
                assert self.config["which_model"] == f"vmamba_{VARIANTE}"
                self.encoder = encoder_validado
                lado = max(self.image_size // 32, DEFAULT_FEATURE_SIZE)
                self.config["feature_hw"] = [lado, lado]

            def get_cls(self):
                assert self.config["cls_name"] == "grid_proto"
                grade = self.config["proto_grid_size"]
                self.cls_unit = MultiProtoAsConv(
                    proto_grid=[grade, grade],
                    feature_hw=self.config["feature_hw"],
                    embed_dim=CANAIS_ENCODER,
                )

            def get_features(self, imgs_concat):
                lado_entrada = self.image_size // 32 * 32
                imgs_concat = F.interpolate(
                    imgs_concat,
                    size=(lado_entrada, lado_entrada),
                    mode="bilinear",
                    align_corners=False,
                )
                features = self.encoder(imgs_concat)[-1]

                assert features.ndim == 4 and features.shape[1] == CANAIS_ENCODER
                if features.shape[-1] < DEFAULT_FEATURE_SIZE:
                    features = F.interpolate(
                        features,
                        size=(DEFAULT_FEATURE_SIZE, DEFAULT_FEATURE_SIZE),
                        mode="bilinear",
                        align_corners=False,
                    )
                return features

        modelo = FewShotSegVMamba(
            image_size=resolucao,
            pretrained_path=None,
            cfg=deepcopy(config),
        )
        return modelo.cuda().eval().requires_grad_(False)

    alpnet = construir_alp_vmamba(
        encoder_validado=encoder,
        resolucao=RESOLUCAO_ENCODER,
        config=CONFIG_ALP,
    )

    assert alpnet.encoder is encoder
    assert type(alpnet).forward is FewShotSeg.forward
    assert all(not m.training for m in alpnet.modules())
    assert all(not p.requires_grad for p in alpnet.parameters())

    print("Encoder já carregado reutilizado:", alpnet.encoder is encoder)
    print("Forward oficial preservado:", type(alpnet).forward is FewShotSeg.forward)
    print("Grade de características:", alpnet.config["feature_hw"])
    print("Kernel configurado do ALP:", alpnet.cls_unit.kernel_size)
    print("Modelo em eval() e congelado.")

## 4. Dados e SAM-H
Mesmos nomes e ordem do manifesto K1. Checkpoints e dados permanecem nos caminhos locais compartilhados, sem downloads.

In [ ]:
if MODO != "consultar":
    import csv
    import inspect
    from dataloaders.PolypDataset import get_polyp_dataset
    from models.segment_anything.utils.transforms import ResizeLongestSide

    REFERENCIA_K1 = (
        RAIZ / "reproducao/01_avaliacao_K1_20261004T140725337013Z"
    )

    assert RAIZ in Path(inspect.getfile(get_polyp_dataset)).resolve().parents

    with (REFERENCIA_K1 / "manifesto_teste.csv").open(
        encoding="utf-8-sig", newline=""
    ) as arquivo:
        manifesto = list(csv.DictReader(arquivo))

    ds_suportes, ds_teste = get_polyp_dataset(
        image_size=(1024, 1024),
        sam_trans=ResizeLongestSide(1024),
    )

    assert len(ds_teste) == len(manifesto) == 100

    # Confere a correspondência com o manifesto, sem executar inferências.
    for item in manifesto:
        indice = int(item["indice"])
        assert Path(ds_teste.images[indice]).resolve() == Path(item["imagem"]).resolve()
        assert Path(ds_teste.gts[indice]).resolve() == Path(item["mascara"]).resolve()
        assert Path(item["imagem"]).stem == Path(item["mascara"]).stem

    indices_k1 = [
        i for i, caminho in enumerate(ds_suportes.images)
        if Path(caminho).name == SUPORTE_K1
    ]
    assert len(indices_k1) == 1
    indice_k1 = indices_k1[0]
    assert Path(ds_suportes.gts[indice_k1]).stem == Path(SUPORTE_K1).stem

    suporte = ds_suportes[indice_k1]
    item_query = manifesto[0]
    query = ds_teste[int(item_query["indice"])]

    assert Path(item_query["imagem"]).name == "cju0u82z3cuma0835wlxrnrjv.png"
    assert int((suporte["label"] > 0.5).sum()) == 51940

    for nome, amostra in (("Suporte K1", suporte), ("Query", query)):
        imagem, mascara = amostra["image"], amostra["label"]
        assert tuple(imagem.shape) == (3, 1024, 1024)
        assert tuple(mascara.shape) == (1024, 1024)
        assert torch.isfinite(imagem).all().item()
        assert ((mascara == 0) | (mascara == 1)).all().item()
        print(
            f"{nome}: imagem={tuple(imagem.shape)}, "
            f"máscara={tuple(mascara.shape)}, "
            f"pixels FG={int(mascara.sum())}"
        )

    print("Índice K1:", indice_k1)
    print("Query:", Path(item_query["imagem"]).name)
    print("Correspondência com o manifesto: conferida.")

In [ ]:
if MODO != "consultar":
    from models.ProtoSAM import ProtoSAM, ALPNetWrapper

    CHECKPOINT_SAM = RAIZ / "pretrained_model/sam_vit_h.pth"
    assert CHECKPOINT_SAM.is_file()
    assert "protosam" not in globals(), (
        "ProtoSAM já instanciado; não execute esta célula novamente."
    )

    CONFIG_SAM = {
        "image_size": [1024, 1024],
        "sam_pretrained_path": str(CHECKPOINT_SAM),
        "num_points_for_sam": 1,
        "use_points": True,
        "use_bbox": True,
        "use_mask": False,
        "debug": False,
        "use_cca": False,
        "point_mode": "both",
        "use_sam_trans": True,
        "coarse_pred_only": False,
        "alpnet_image_size": RESOLUCAO_ENCODER,
        "use_neg_points": False,
    }

    wrapper = ALPNetWrapper(alpnet)
    protosam = ProtoSAM(
        coarse_segmentation_model=wrapper,
        **CONFIG_SAM,
    ).to("cuda", dtype=torch.float32)

    protosam.eval().requires_grad_(False)
    alpnet.eval().requires_grad_(False)

    assert not any(p.requires_grad for p in alpnet.parameters())
    assert not any(p.requires_grad for p in protosam.sam.parameters())
    assert not alpnet.training
    assert not protosam.training
    assert not protosam.sam.training

    print("SAM-H carregado:", CHECKPOINT_SAM.resolve())
    print(f"Encoder: VMamba-{VARIANTE} congelado; {CANAIS_ENCODER} canais")
    print("Prompts: caixa + ponto de confiança + centroide por componente")
    print("CCA: False")
    print("Canvas SAM: 1024 × 1024 | entrada VMamba: 672 × 672")

## 5. Função de avaliação de uma query
ALP é executado separadamente e recalculado dentro do pipeline SAM, como na inspeção. Não se cronometra este caminho como benchmark. As chamadas CUDA são contadas durante a inferência real e o fallback PyTorch é recusado.

In [ ]:
if MODO != "consultar":
    def avaliar_item(item):
        amostra = ds_teste[int(item["indice"])]
        assert tuple(amostra["image"].shape) == (3, 1024, 1024)
        assert torch.isfinite(amostra["image"]).all().item()
        mascara_gt = amostra["label"].detach().cpu().numpy()
        assert np.isin(mascara_gt, [0, 1]).all()
        gt = mascara_gt > .5
        entrada = ALPNetInput(
            support_images=[suporte["image"].unsqueeze(0).cuda().float()],
            support_labels=[suporte["label"].unsqueeze(0).cuda().float()],
            query_images=amostra["image"].unsqueeze(0).cuda().float(),
            isval=True, val_wsize=2, show_viz=False, supp_fts=None)
        qry_local = amostra["image"].unsqueeze(0).cuda().float()
        contagem = {"cuda": 0, "fallback_torch": 0}
        assert vmamba.WITH_SELECTIVESCAN_MAMBA, "Extensão selective scan ausente"
        ext = vmamba.selective_scan_cuda
        original_cuda, original_torch = ext.fwd, vmamba.selective_scan_torch
        def contar_cuda(*args, **kwargs):
            contagem["cuda"] += 1
            return original_cuda(*args, **kwargs)
        def recusar_fallback(*args, **kwargs):
            contagem["fallback_torch"] += 1
            raise RuntimeError("Fallback selective_scan_torch não permitido neste protocolo")
        try:
            ext.fwd, vmamba.selective_scan_torch = contar_cuda, recusar_fallback
            with torch.inference_mode(), torch.autocast("cuda", enabled=False):
                logits = wrapper(entrada)
                assert tuple(logits.shape) == (1, 2, 1024, 1024)
                assert torch.isfinite(logits).all().item()
                alp = (logits.argmax(dim=1)[0] == 1).cpu().numpy()
                final, scores = protosam(query_image=qry_local,
                    coarse_model_input=entrada, degrees_rotate=0)
            torch.cuda.synchronize()
        finally:
            ext.fwd, vmamba.selective_scan_torch = original_cuda, original_torch
        assert contagem["cuda"] == 2 * sum(ARQUITETURA["depths"]) and contagem["fallback_torch"] == 0, contagem
        sam = final.detach().cpu().numpy() if isinstance(final, torch.Tensor) else np.asarray(final)
        sam = sam.squeeze()
        assert sam.shape == (1024, 1024) and np.isin(sam, [0, 1]).all()
        sam = sam.astype(bool)
        return {"alp": alp, "sam": sam, "gt": gt}, {
            "indice": int(item["indice"]), "query": Path(item["imagem"]).name,
            "metricas": {"ALP": metricas_binarias(alp, gt), "SAM": metricas_binarias(sam, gt)},
            "backend": contagem}


## 6. Identidade da execução e retomada
Hashes cobrem dados, pesos, fontes Python locais e dependências relevantes. O JSON por imagem é o registro de conclusão; CSV e resumo são reconstruídos a partir dele. Alterar GPU, driver ou código impede retomar a mesma pasta. Uma interrupção entre máscaras e JSON deixa a imagem pendente, para reexecução.

In [ ]:
if MODO != "consultar":
    import importlib.metadata as metadata
    import segment_anything
    assert NOTEBOOK_ATUAL.is_file(), "Salve o notebook no caminho NOTEBOOK_ATUAL antes de executar"
    nb_atual = ler_json(NOTEBOOK_ATUAL)
    fontes = ["".join(c["source"]) for c in nb_atual["cells"]
              if c["cell_type"] == "code" and c.get("metadata", {}).get("papel") not in ("config", "historico")]
    versoes = {}
    for nome in ("torch", "torchvision", "numpy", "Pillow", "timm", "triton", "mamba-ssm", "fvcore", "scipy", "scikit-image", "opencv-python", "segment-anything"):
        try: versoes[nome] = metadata.version(nome)
        except metadata.PackageNotFoundError: versoes[nome] = "não registrado"
    ambiente = {
        "python": sys.version, "executavel": sys.executable,
        "gpu": torch.cuda.get_device_name(0), "capacidade": torch.cuda.get_device_capability(0),
        "driver": subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True, check=True).stdout.strip(),
        "versoes": versoes, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "threads": torch.get_num_threads(), "threads_interop": torch.get_num_interop_threads(),
        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32, "tf32_cudnn": torch.backends.cudnn.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "deterministic": torch.are_deterministic_algorithms_enabled(),
        "matmul_precision": torch.get_float32_matmul_precision(),
        "triton_importado": vmamba.WITH_TRITON,
        "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    }
    arquivos_codigo = {}
    for subdir in ("models", "dataloaders", "util"):
        for p in sorted((RAIZ / subdir).rglob("*.py")):
            arquivos_codigo[str(p.relative_to(RAIZ))] = sha256_arquivo(p)
    for p in sorted(Path(segment_anything.__file__).parent.rglob("*.py")):
        arquivos_codigo["pacote_sam/" + str(p.relative_to(Path(segment_anything.__file__).parent))] = sha256_arquivo(p)
    arquivos_codigo["selective_scan_cuda"] = sha256_arquivo(vmamba.selective_scan_cuda.__file__)
    manifesto_hashes = [dict(item, sha_imagem=sha256_arquivo(item["imagem"]),
                             sha_mascara=sha256_arquivo(item["mascara"])) for item in manifesto]
    selecionados = manifesto_hashes[:1] if MODO == "inspecionar" else manifesto_hashes
    assinatura = canonico({
        "schema": 2, "variante": VARIANTE, "canais_encoder": CANAIS_ENCODER, "modo": MODO, "ambiente": ambiente, "semente": 42,
        "config_vmamba": CONFIG_VMAMBA, "config_alp": alpnet.config, "config_sam": CONFIG_SAM,
        "resolucao_encoder": RESOLUCAO_ENCODER, "normalizacao_saida": "escala1_vies0_congelada",
        "val_wsize": 2, "isval": True, "rotacao": 0, "cache_suporte": False,
        "fontes_notebook": hashlib.sha256("\n".join(fontes).encode()).hexdigest(),
        "codigo": arquivos_codigo,
        "pesos": {"vmamba": sha256_arquivo(CHECKPOINT_VMAMBA), "sam": sha256_arquivo(CHECKPOINT_SAM)},
        "suporte": {"nome": SUPORTE_K1, "imagem": sha256_arquivo(ds_suportes.images[indice_k1]),
                    "mascara": sha256_arquivo(ds_suportes.gts[indice_k1])},
        "dados": manifesto_hashes,
        
    })
    if RETOMAR_DIR is None:
        DIR_EXECUCAO = RAIZ / "reproducao" / (f"vmamba_{VARIANTE}_k1_" + MODO + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
        DIR_EXECUCAO.mkdir(parents=True, exist_ok=False)
        salvar_json(DIR_EXECUCAO / "assinatura.json", assinatura)
        (DIR_EXECUCAO / "notebook_executado.ipynb").write_bytes(NOTEBOOK_ATUAL.read_bytes())
        (DIR_EXECUCAO / "manifesto_teste.csv").write_bytes((REFERENCIA_K1 / "manifesto_teste.csv").read_bytes())
        for sub in ("alp", "sam", "gt", "resultados"): (DIR_EXECUCAO / sub).mkdir()
    else:
        DIR_EXECUCAO = Path(RETOMAR_DIR)
        assert ler_json(DIR_EXECUCAO / "assinatura.json") == assinatura, "Retomada recusada: ambiente, código, configuração, dados ou pesos diferem"
    print("Diretório da execução:", DIR_EXECUCAO)


In [ ]:
def resultados_validos(pasta, itens):
    nomes = {Path(item["imagem"]).stem: item for item in itens}
    assert len(nomes) == len(itens), "Nomes de queries duplicados"
    encontrados = {p.stem: p for p in (pasta / "resultados").glob("*.json")}
    assert set(encontrados) <= set(nomes), "Registro de query fora do manifesto"
    resultados = []
    for chave, item in nomes.items():
        if chave not in encontrados: continue
        r = ler_json(encontrados[chave])
        assert r["query"] == Path(item["imagem"]).name and r["indice"] == int(item["indice"])
        for tipo in ("alp", "sam", "gt"):
            p = pasta / tipo / (chave + ".png")
            assert sha256_arquivo(p) == r["hashes_mascaras"][tipo], f"Máscara alterada: {p}"
        gt = ler_mascara(pasta / "gt" / (chave + ".png"))
        for tipo in ("alp", "sam"):
            assert metricas_binarias(ler_mascara(pasta / tipo / (chave + ".png")), gt) == r["metricas"][tipo.upper()]
        resultados.append(r)
    return resultados

def atualizar_relatorios(pasta, resultados, previstas, estado, erro=None):
    colunas = ["indice", "query"] + [f"{m}_{k}" for m in ("ALP", "SAM") for k in ("Dice", "IoU", "precisao", "recall", "TP", "FP", "FN")] + ["delta_Dice_SAM_menos_ALP"]
    tmp = pasta / "por_imagem.csv.tmp"
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=colunas); w.writeheader()
        for r in resultados:
            linha = {k: r[k] for k in ("indice", "query")}
            linha.update({f"{m}_{k}": v for m, metricas in r["metricas"].items() for k, v in metricas.items()})
            linha["delta_Dice_SAM_menos_ALP"] = r["metricas"]["SAM"]["Dice"] - r["metricas"]["ALP"]["Dice"]
            w.writerow(linha)
    os.replace(tmp, pasta / "por_imagem.csv")
    resumo = {"n": len(resultados), "previstas": previstas, "parcial": len(resultados) != previstas}
    if resultados:
        resumo["medias_por_imagem"] = {m: {k: float(np.mean([r["metricas"][m][k] for r in resultados])) for k in ("Dice", "IoU", "precisao", "recall")} for m in ("ALP", "SAM")}
        delta = np.array([r["metricas"]["SAM"]["Dice"] - r["metricas"]["ALP"]["Dice"] for r in resultados])
        resumo.update(delta_medio=float(delta.mean()), delta_mediano=float(np.median(delta)),
            melhora=int((delta > 1e-6).sum()), piora=int((delta < -1e-6).sum()), empate=int((np.abs(delta) <= 1e-6).sum()))
    salvar_json(pasta / "resumo.json", resumo)
    salvar_json(pasta / "status.json", dict(status=estado, previstas=previstas,
        avaliadas=len(resultados), pendentes=previstas-len(resultados), excluidas=0, erro=erro))


In [ ]:
import torch

assert VARIANTE == "base"
assert alpnet.encoder is encoder
assert not alpnet.training and not protosam.training

print("GPU:", torch.cuda.get_device_name())
print("Dimensões por estágio:", encoder.dims)
print("Blocos por estágio:", [
    len(estagio.blocks) for estagio in encoder.layers
])
print("Estágios retornados:", encoder.out_indices)
print("Queries:", len(ds_teste))
print("Suporte:", SUPORTE_K1)

## 7. Executar ou retomar
No modo inspecionar esta célula processa somente a primeira query; no modo avaliar processa as 100 queries. Erros interrompem a execução e ficam registrados; imagens não são descartadas. Para retomar, reinicie o kernel e use o caminho impresso acima. Não execute dois kernels na mesma pasta.

In [ ]:
if MODO != "consultar":
    import fcntl
    with (DIR_EXECUCAO / "execucao.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        resultados = resultados_validos(DIR_EXECUCAO, selecionados)
        prontas = {r["query"] for r in resultados}
        atualizar_relatorios(DIR_EXECUCAO, resultados, len(selecionados), "executando")
        try:
            for item in selecionados:
                nome = Path(item["imagem"]).name
                if nome in prontas: continue
                mascaras, r = avaliar_item(item)
                r["hashes_mascaras"] = {}
                for tipo, mascara in mascaras.items():
                    destino = DIR_EXECUCAO / tipo / (Path(nome).stem + ".png")
                    tmp = destino.with_suffix(".png.tmp")
                    Image.fromarray(mascara.astype(np.uint8)*255).save(tmp, format="PNG")
                    os.replace(tmp, destino)
                    r["hashes_mascaras"][tipo] = sha256_arquivo(destino)
                salvar_json(DIR_EXECUCAO / "resultados" / (Path(nome).stem + ".json"), r)
                resultados.append(r)
                atualizar_relatorios(DIR_EXECUCAO, resultados, len(selecionados), "executando")
                print(f"[{len(resultados)}/{len(selecionados)}] {nome}: ALP={r['metricas']['ALP']['Dice']:.6f} SAM={r['metricas']['SAM']['Dice']:.6f}", flush=True)
            atualizar_relatorios(DIR_EXECUCAO, resultados, len(selecionados), "concluida")
        except BaseException as exc:
            estado = "interrompida" if isinstance(exc, KeyboardInterrupt) else "erro"
            atualizar_relatorios(DIR_EXECUCAO, resultados, len(selecionados), estado, repr(exc))
            raise
    print(json.dumps(ler_json(DIR_EXECUCAO / "resumo.json"), indent=2, ensure_ascii=False))


## 8. Inspeção visual da primeira query
Lê as máscaras salvas, sem nova inferência. A desnormalização RGB serve apenas à figura.

In [ ]:
if MODO != "consultar":
    primeiro = selecionados[0]
    nome = Path(primeiro["imagem"]).stem + ".png"
    resultado = ler_json(DIR_EXECUCAO / "resultados" / (Path(nome).stem + ".json"))
    amostra = ds_teste[int(primeiro["indice"])]
    media = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1)
    desvio = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1)
    rgb = (amostra["image"].cpu() * desvio + media).clamp(0, 255).permute(1, 2, 0).numpy().astype(np.uint8)
    paineis = [(rgb, "Query"), (ler_mascara(DIR_EXECUCAO / "gt" / nome), "Máscara humana")]
    for m in ("alp", "sam"):
        paineis.append((ler_mascara(DIR_EXECUCAO / m / nome), f"{VARIANTE} {m.upper()} — Dice {resultado['metricas'][m.upper()]['Dice']:.4f}"))
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (im, titulo) in zip(axes, paineis):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1) if im.ndim == 2 else ax.imshow(im)
        ax.set_title(titulo); ax.axis("off")
    fig.tight_layout()
    fig.savefig(DIR_EXECUCAO / "comparacao.png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    display(DisplayImage(filename=str(DIR_EXECUCAO / "comparacao.png")))
    print("Salvo:", DIR_EXECUCAO)


In [ ]:
print('Dire execução', DIR_EXECUCAO)

In [ ]:
from pathlib import Path
import sys
import inspect

REPO = Path("/workspace/ProtoSAM-repro")
assert (REPO / "models" / "ProtoSAM.py").is_file(), (
    f"Repositório não encontrado: {REPO}"
)

sys.path.insert(0, str(REPO))
from models.ProtoSAM import ProtoSAM

origem = Path(inspect.getfile(ProtoSAM)).resolve()
assert origem == (REPO / "models" / "ProtoSAM.py").resolve(), (
    f"Módulo carregado de outro local: {origem}"
)

for nome in (
    "get_bbox_per_cc",
    "get_sam_input_points",
    "predict_w_points_bbox",
):
    print(f"\n{'=' * 20} {nome} {'=' * 20}\n")
    print(inspect.getsource(getattr(ProtoSAM, nome)))

In [ ]:
import inspect

print("=== Seleção do ponto de confiança ===")
print(inspect.getsource(ProtoSAM.get_most_conf_points))

print("\n=== Objetos disponíveis no kernel ===")
for nome in (
    "protosam",
    "alpnet",
    "encoder",
    "ds_test",
    "sup_img",
    "sup_mask",
):
    objeto = globals().get(nome)
    if objeto is None:
        print(f"{nome}: ausente")
    else:
        classe = type(objeto)
        print(f"{nome}: {classe.__module__}.{classe.__name__}")

modelo = globals().get("protosam")
if modelo is not None:
    print("\n=== Configuração do ProtoSAM carregado ===")
    for atributo in (
        "training", "use_cca", "point_mode",
        "use_bbox", "use_points", "use_neg_points",
    ):
        print(f"{atributo}: {getattr(modelo, atributo, 'não encontrado')}")

In [ ]:
from pathlib import Path
import sys
import subprocess
import torch

REPO = Path("/workspace/ProtoSAM-repro")
BASE_4090 = (
    REPO / "reproducao"
    / "vmamba_base_k1_avaliar_20261008T134621894550Z"
)
QUERY = "cju3tp94kfstl08181awh6z49.png"

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())

resultado = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)
print("GPU:", resultado.stdout.strip() or resultado.stderr.strip())

arquivos = {
    "Código ProtoSAM": REPO / "models/ProtoSAM.py",
    "Código VMamba": REPO / "models/vmamba.py",
    "Checkpoint Base": REPO / "pretrained_model/vmamba_base_v2.pth",
    "Checkpoint SAM": REPO / "pretrained_model/sam_vit_h.pth",
    "Resultados Base 4090": BASE_4090 / "por_imagem.csv",
    "Máscara ALP 4090": BASE_4090 / "alp" / QUERY,
    "Máscara SAM 4090": BASE_4090 / "sam" / QUERY,
}

for descricao, caminho in arquivos.items():
    print(f"{descricao}: {'OK' if caminho.is_file() else 'AUSENTE'}")

In [ ]:
for nome in (
    "protosam", "alpnet", "encoder",
    "ds_test", "sup_img", "sup_mask",
):
    objeto = globals().get(nome)
    print(
        f"{nome}: "
        f"{'disponível' if objeto is not None else 'ausente'}"
    )

In [ ]:
termos = ("dataset", "support", "suporte", "query", "sup_", "ds_", "teste")

for nome, objeto in list(globals().items()):
    if nome.startswith("_"):
        continue
    if any(termo in nome.lower() for termo in termos):
        classe = type(objeto)
        print(f"{nome}: {classe.__module__}.{classe.__name__}")

In [ ]:
print("Suporte K1 configurado:", SUPORTE_K1)
print("Quantidade de queries:", len(ds_teste))

def descrever_item(nome, item):
    print(f"\n=== {nome} ===")
    for chave, valor in item.items():
        if hasattr(valor, "shape"):
            print(
                f"{chave}: shape={tuple(valor.shape)}, "
                f"dtype={getattr(valor, 'dtype', None)}"
            )
        elif isinstance(valor, (str, int, float, bool, type(None))):
            print(f"{chave}: {valor}")
        else:
            print(f"{chave}: {type(valor).__name__}")

descrever_item("suporte", suporte)
descrever_item("primeira query do dataset", ds_teste[0])

In [ ]:
from pathlib import Path
import pandas as pd

BASE_4090 = Path(
    "/workspace/ProtoSAM-repro/reproducao/"
    "vmamba_base_k1_avaliar_20261008T134621894550Z"
)
QUERY_DIAGNOSTICO = "cju3tp94kfstl08181awh6z49.png"

resultados = pd.read_csv(BASE_4090 / "por_imagem.csv")
registro = resultados.loc[resultados["query"] == QUERY_DIAGNOSTICO]
assert len(registro) == 1, "Query ausente ou duplicada."

INDICE_DIAGNOSTICO = int(registro.iloc[0]["indice"])
print("Query:", QUERY_DIAGNOSTICO)
print("Índice registrado:", INDICE_DIAGNOSTICO)
display(registro[["query", "ALP_Dice", "SAM_Dice"]])

print("\n=== Estrutura do dataset ===")
for nome, valor in vars(ds_teste).items():
    if isinstance(valor, (str, Path)):
        print(f"{nome}: {valor}")
    elif isinstance(valor, (list, tuple)):
        print(f"{nome}: {type(valor).__name__}, tamanho={len(valor)}")
        if len(valor) > INDICE_DIAGNOSTICO:
            elemento = valor[INDICE_DIAGNOSTICO]
            print(f"  item[{INDICE_DIAGNOSTICO}]: {str(elemento)[:500]}")
    elif isinstance(valor, dict):
        print(f"{nome}: dict, chaves={list(valor)[:15]}")

In [ ]:
import numpy as np
import torch
from PIL import Image
from pathlib import Path
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

assert Path(ds_teste.images[INDICE_DIAGNOSTICO]).name == QUERY_DIAGNOSTICO
assert not protosam.training and not alpnet.training
assert protosam.use_cca is False

device = next(alpnet.parameters()).device
item_diagnostico = ds_teste[INDICE_DIAGNOSTICO]

entrada_diagnostico = ALPNetInput(
    support_images=[suporte["image"].unsqueeze(0).to(device)],
    support_labels=[suporte["label"].unsqueeze(0).to(device)],
    query_images=item_diagnostico["image"].unsqueeze(0).to(device),
    isval=True,
    val_wsize=2,
    show_viz=False,
    supp_fts=None,
)
imagem_diagnostico = item_diagnostico["image"].unsqueeze(0).to(device)
gt_diagnostico = item_diagnostico["label"].cpu().numpy() > 0.5

capturas_sam = []
imagem_recebida_sam = {}
predict_original = protosam.predictor.predict
set_image_original = protosam.predictor.set_image

def copiar_array(valor):
    return None if valor is None else np.array(valor, copy=True)

def set_image_observado(imagem, *args, **kwargs):
    imagem_recebida_sam["rgb"] = copiar_array(imagem)
    return set_image_original(imagem, *args, **kwargs)

def predict_observado(*args, **kwargs):
    saida = predict_original(*args, **kwargs)
    mascaras, scores, _ = saida
    capturas_sam.append({
        "pontos": copiar_array(kwargs.get("point_coords")),
        "labels": copiar_array(kwargs.get("point_labels")),
        "caixa": copiar_array(kwargs.get("box")),
        "mascaras": copiar_array(mascaras),
        "scores": copiar_array(scores),
    })
    return saida  # Preserva todas as saídas e a seleção oficial.

try:
    protosam.predictor.predict = predict_observado
    protosam.predictor.set_image = set_image_observado

    with torch.inference_mode():
        logits = ALPNetWrapper(alpnet)(entrada_diagnostico)
        alp_diagnostico = logits.argmax(dim=1)[0].cpu().numpy() > 0

        saida_sam, _ = protosam(
            query_image=imagem_diagnostico,
            coarse_model_input=entrada_diagnostico,
            degrees_rotate=0,
        )
        sam_diagnostico = saida_sam.cpu().numpy() > 0

finally:
    protosam.predictor.predict = predict_original
    protosam.predictor.set_image = set_image_original

def dice_binario(pred, gt):
    denominador = int(pred.sum()) + int(gt.sum())
    return (
        2 * np.count_nonzero(pred & gt) / denominador
        if denominador else 1.0
    )

print("GPU:", torch.cuda.get_device_name(device))
print("Query:", QUERY_DIAGNOSTICO)
print("Chamadas ao SAM:", len(capturas_sam))

for etapa, atual in (
    ("alp", alp_diagnostico),
    ("sam", sam_diagnostico),
):
    with Image.open(BASE_4090 / etapa / QUERY_DIAGNOSTICO) as imagem:
        anterior = np.asarray(imagem.convert("L")) > 0

    assert atual.shape == anterior.shape == gt_diagnostico.shape
    print(
        f"{etapa.upper()}: "
        f"Dice 5090={dice_binario(atual, gt_diagnostico):.6f} | "
        f"pixels divergentes da 4090={np.count_nonzero(atual != anterior)}"
    )

assert capturas_sam, "Nenhuma chamada ao SAM foi capturada."
print("Alternativas por chamada:",
      [len(c["mascaras"]) for c in capturas_sam])

In [ ]:
import pandas as pd
import numpy as np
from io import BytesIO
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Rectangle
from IPython.display import display, Image as ImagemNotebook

fig = Figure(figsize=(18, 4.5 * len(capturas_sam)))
canvas = FigureCanvasAgg(fig)
eixos = fig.subplots(len(capturas_sam), 4, squeeze=False)
linhas = []

for i, captura in enumerate(capturas_sam):
    eixo = eixos[i, 0]
    eixo.imshow(imagem_recebida_sam["rgb"])
    eixo.contour(
        gt_diagnostico.astype(float),
        levels=[0.5], colors=["cyan"], linewidths=1,
    )

    caixa = captura["caixa"]
    if caixa is not None:
        x0, y0, x1, y1 = np.asarray(caixa).reshape(-1)
        eixo.add_patch(Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            fill=False, edgecolor="yellow", linewidth=2,
        ))

    pontos = captura["pontos"]
    if pontos is not None:
        for numero, (x, y) in enumerate(pontos):
            eixo.scatter(x, y, c="lime", marker="*", s=130)
            eixo.annotate(
                str(numero), (x, y),
                xytext=(5, 5), textcoords="offset points",
                color="black",
                bbox=dict(facecolor="white", alpha=0.8),
            )

    eixo.set_title(
        f"Componente {i + 1}: prompts\n"
        "0: confiança | 1: centroide | ciano: referência"
    )

    scores = np.asarray(captura["scores"]).reshape(-1)
    maior_score = int(scores.argmax())

    for j, mascara in enumerate(captura["mascaras"]):
        mascara = np.asarray(mascara) > 0
        assert mascara.shape == gt_diagnostico.shape
        dice = dice_binario(mascara, gt_diagnostico)

        painel = np.zeros((*gt_diagnostico.shape, 3), dtype=np.uint8)
        painel[mascara & gt_diagnostico] = [255, 255, 255]
        painel[mascara & ~gt_diagnostico] = [230, 65, 65]
        painel[~mascara & gt_diagnostico] = [65, 130, 255]

        marcas = []
        if j == 0:
            marcas.append("seleção original")
        if j == maior_score:
            marcas.append("maior score")

        eixos[i, j + 1].imshow(painel)
        eixos[i, j + 1].set_title(
            f"Alternativa {j} — {', '.join(marcas)}\n"
            f"Score SAM={scores[j]:.4f} | Dice={dice:.4f}"
        )
        linhas.append({
            "componente": i + 1,
            "alternativa": j,
            "score_SAM": float(scores[j]),
            "Dice": dice,
            "seleção_original": j == 0,
        })

    for eixo in eixos[i]:
        eixo.axis("off")

fig.tight_layout()
buffer = BytesIO()
canvas.print_png(buffer)
display(ImagemNotebook(data=buffer.getvalue()))
buffer.close()

display(pd.DataFrame(linhas).round(6))

In [ ]:
regras = {
    "Original: alternativa 0": [
        0 for captura in capturas_sam
    ],
    "Maior score por componente": [
        int(np.argmax(captura["scores"]))
        for captura in capturas_sam
    ],
}

for nome, indices in regras.items():
    uniao = np.logical_or.reduce([
        np.asarray(captura["mascaras"][indice]) > 0
        for captura, indice in zip(capturas_sam, indices)
    ])

    if nome == "Original: alternativa 0":
        assert np.array_equal(uniao, sam_diagnostico), (
            "A união reconstruída difere da saída original."
        )

    print(
        f"{nome}\n"
        f"  Alternativas: {indices}\n"
        f"  Dice da união: {dice_binario(uniao, gt_diagnostico):.6f}\n"
        f"  Pixels preditos: {int(uniao.sum())}\n"
    )

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import inspect
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from models.ProtoSAM import ALPNetInput

assert VARIANTE == "base", "Esta célula foi preparada para o Base."
assert not protosam.training and not alpnet.training
assert protosam.use_cca is False
assert len(ds_teste) == 100

# Confere o pareamento com a avaliação anterior.
referencia = pd.read_csv(BASE_4090 / "por_imagem.csv")
for _, linha in referencia.iterrows():
    indice = int(linha["indice"])
    assert Path(ds_teste.images[indice]).name == linha["query"]

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_COMPARACAO = Path("/workspace/ProtoSAM-repro/reproducao") / (
    f"vmamba_base_k1_selecao_sam_exploratoria_{timestamp}"
)
DIR_COMPARACAO.mkdir(parents=True, exist_ok=False)
for nome in ("original", "maior_score", "registros"):
    (DIR_COMPARACAO / nome).mkdir()

def gravar_json(caminho, dados):
    temporario = caminho.with_suffix(caminho.suffix + ".tmp")
    temporario.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    temporario.replace(caminho)

codigo = inspect.getsource(type(protosam).predict_w_points_bbox)
gravar_json(DIR_COMPARACAO / "config.json", {
    "tipo": "analise exploratoria pareada",
    "variante": VARIANTE,
    "suporte": SUPORTE_K1,
    "gpu": torch.cuda.get_device_name(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "referencia_4090": str(BASE_4090),
    "regras": ["alternativa_0", "argmax_score_por_componente"],
    "use_cca": protosam.use_cca,
    "point_mode": protosam.point_mode,
    "use_bbox": protosam.use_bbox,
    "use_neg_points": protosam.use_neg_points,
    "codigo_selecao_sha256": hashlib.sha256(codigo.encode()).hexdigest(),
})
(DIR_COMPARACAO / "selecao_original.py").write_text(codigo)

device = next(alpnet.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)

predict_original = protosam.predictor.predict
acumulador = {}

def observar_predict(*args, **kwargs):
    saida = predict_original(*args, **kwargs)
    mascaras, scores, _ = saida

    assert not kwargs.get("return_logits", False)
    mascaras = np.asarray(mascaras, dtype=bool)
    scores = np.asarray(scores).reshape(-1)
    assert len(mascaras) == len(scores)
    assert np.isfinite(scores).all()

    melhor = int(scores.argmax())
    acumulador["original"] |= mascaras[0]
    acumulador["maior_score"] |= mascaras[melhor]
    acumulador["componentes"].append({
        "scores": scores.tolist(),
        "indice_maior_score": melhor,
    })
    return saida  # Mantém o comportamento original do pipeline.

def dice(pred, gt):
    total = int(pred.sum()) + int(gt.sum())
    return 2 * int((pred & gt).sum()) / total if total else 1.0

linhas = []
estado = "executando"
erro = None
print("Diretório:", DIR_COMPARACAO)

try:
    protosam.predictor.predict = observar_predict

    with torch.inference_mode():
        for _, registro in referencia.sort_values("indice").iterrows():
            indice = int(registro["indice"])
            nome = registro["query"]
            item = ds_teste[indice]
            gt = item["label"].cpu().numpy() > 0.5

            acumulador = {
                "original": np.zeros_like(gt, dtype=bool),
                "maior_score": np.zeros_like(gt, dtype=bool),
                "componentes": [],
            }

            qry = item["image"].unsqueeze(0).to(device)
            entrada = ALPNetInput(
                support_images=[sup_img],
                support_labels=[sup_mask],
                query_images=qry,
                isval=True,
                val_wsize=2,
                show_viz=False,
                supp_fts=None,
            )

            saida, _ = protosam(
                query_image=qry,
                coarse_model_input=entrada,
                degrees_rotate=0,
            )
            original = saida.cpu().numpy() > 0

            if acumulador["componentes"]:
                assert np.array_equal(
                    original, acumulador["original"]
                ), f"União reconstruída diferente: {nome}"
                alternativa = acumulador["maior_score"]
            else:
                # Sem foreground ALP, o pipeline não chama o SAM.
                alternativa = original.copy()

            d_original = dice(original, gt)
            d_alternativa = dice(alternativa, gt)
            linha = {
                "indice": indice,
                "query": nome,
                "Dice_original": d_original,
                "Dice_maior_score": d_alternativa,
                "delta": d_alternativa - d_original,
                "componentes": len(acumulador["componentes"]),
            }

            for pasta, mascara in (
                ("original", original),
                ("maior_score", alternativa),
            ):
                Image.fromarray(
                    mascara.astype(np.uint8) * 255
                ).save(DIR_COMPARACAO / pasta / nome)

            gravar_json(
                DIR_COMPARACAO / "registros" / f"{Path(nome).stem}.json",
                {**linha, "alternativas": acumulador["componentes"]},
            )
            linhas.append(linha)
            pd.DataFrame(linhas).to_csv(
                DIR_COMPARACAO / "por_imagem.csv", index=False
            )

            print(
                f"[{len(linhas)}/100] {nome}: "
                f"original={d_original:.6f} "
                f"maior_score={d_alternativa:.6f} "
                f"delta={linha['delta']:+.6f}"
            )

    estado = "concluida"

except BaseException as exc:
    estado = "interrompida"
    erro = repr(exc)
    raise

finally:
    protosam.predictor.predict = predict_original
    gravar_json(DIR_COMPARACAO / "status.json", {
        "status": estado,
        "avaliadas": len(linhas),
        "previstas": 100,
        "erro": erro,
    })

df_comparacao = pd.DataFrame(linhas)
resumo = {
    "n": len(df_comparacao),
    "Dice_original": float(df_comparacao["Dice_original"].mean()),
    "Dice_maior_score": float(df_comparacao["Dice_maior_score"].mean()),
    "delta_medio": float(df_comparacao["delta"].mean()),
    "delta_mediano": float(df_comparacao["delta"].median()),
    "melhora": int((df_comparacao["delta"] > 0).sum()),
    "piora": int((df_comparacao["delta"] < 0).sum()),
    "empate": int((df_comparacao["delta"] == 0).sum()),
}
gravar_json(DIR_COMPARACAO / "resumo.json", resumo)
print(json.dumps(resumo, indent=2))

In [ ]:
from pathlib import Path
import ast

arquivo = Path("/workspace/ProtoSAM-repro/models/vmamba.py")
assert arquivo.is_file(), arquivo

codigo = arquivo.read_text(encoding="utf-8")
linhas = codigo.splitlines()
arvore = ast.parse(codigo)

def mostrar(no):
    print("\n".join(linhas[no.lineno - 1:no.end_lineno]))

# Implementação da normalização para tensores de imagem.
for no in arvore.body:
    if isinstance(no, ast.ClassDef) and no.name == "LayerNorm2d":
        print("\n=== LayerNorm2d ===")
        mostrar(no)

# Caminho das características no classificador e no backbone.
for no in arvore.body:
    if not isinstance(no, ast.ClassDef):
        continue
    if no.name not in ("VSSM", "Backbone_VSSM"):
        continue

    for metodo in no.body:
        if not isinstance(metodo, ast.FunctionDef):
            continue

        if metodo.name in ("forward", "forward_features"):
            print(f"\n=== {no.name}.{metodo.name} ===")
            mostrar(metodo)

        elif metodo.name == "__init__":
            if no.name == "Backbone_VSSM":
                print("\n=== Backbone_VSSM.__init__ ===")
                mostrar(metodo)
            else:
                print("\n=== VSSM: construção da normalização/cabeça ===")
                # Mostra blocos do construtor que definem esses elementos.
                for bloco in metodo.body:
                    trecho = "\n".join(
                        linhas[bloco.lineno - 1:bloco.end_lineno]
                    )
                    if any(termo in trecho for termo in (
                        "self.classifier", "self.norm_layer",
                        "_NORMLAYERS", "self.channel_first",
                    )):
                        print(trecho)

In [ ]:
from pathlib import Path
import ast
import torch

REPO = Path("/workspace/ProtoSAM-repro")
arquivo = REPO / "models/vmamba.py"
codigo = arquivo.read_text(encoding="utf-8")
linhas = codigo.splitlines()

print("=== Implementação de LayerNorm ===")
for no in ast.parse(codigo).body:
    if isinstance(no, ast.ClassDef) and no.name == "LayerNorm":
        print("\n".join(linhas[no.lineno - 1:no.end_lineno]))

for variante in ("tiny", "small", "base"):
    caminho = REPO / "pretrained_model" / f"vmamba_{variante}_v2.pth"
    checkpoint = torch.load(
        caminho, map_location="cpu", weights_only=True
    )
    pesos = checkpoint["model"]

    print(f"\n=== {variante.upper()} — classifier.norm ===")
    for parametro, inicial in (("weight", 1.0), ("bias", 0.0)):
        chave = f"classifier.norm.{parametro}"
        tensor = pesos[chave].detach().float()
        diferenca = (tensor - inicial).abs()

        print(
            f"{parametro}: shape={tuple(tensor.shape)} | "
            f"min={tensor.min().item():.6f} | "
            f"max={tensor.max().item():.6f} | "
            f"média={tensor.mean().item():.6f} | "
            f"desvio médio da inicialização={diferenca.mean().item():.6f}"
        )

    del pesos, checkpoint

In [ ]:
modelo = globals().get("alpnet")

if modelo is None:
    print("ALP não está carregado neste kernel.")
else:
    enc = modelo.encoder
    print("Encoder:", type(enc).__name__)
    print("Device:", next(enc.parameters()).device)
    print("Dimensões por estágio:", getattr(enc, "dims", None))
    print("Estágios de saída:", getattr(enc, "out_indices", None))

    norma = enc.outnorm3
    print("Shape de outnorm3:", tuple(norma.weight.shape))
    print("Epsilon:", norma.eps)
    print(
        "Inicialização padrão:",
        bool((norma.weight == 1).all().item())
        and bool((norma.bias == 0).all().item()),
    )

print("VARIANTE:", globals().get("VARIANTE", "não definida"))
print("Dataset disponível:", "ds_teste" in globals())
print("Suporte disponível:", "suporte" in globals())

In [ ]:
from pathlib import Path
import torch

assert VARIANTE == "base"
assert alpnet.encoder is encoder
assert not encoder.training
assert "outnorm3_antes" not in globals(), (
    "Backup já existe. Não repita esta célula sem conferir o estado."
)

norma = encoder.outnorm3
assert torch.all(norma.weight == 1).item()
assert torch.all(norma.bias == 0).item()

checkpoint_path = Path(
    "/workspace/ProtoSAM-repro/pretrained_model/vmamba_base_v2.pth"
)
checkpoint = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)

novos_pesos = {
    nome: checkpoint["model"][f"classifier.norm.{nome}"].detach().clone()
    for nome in ("weight", "bias")
}

for nome, tensor in novos_pesos.items():
    atual = getattr(norma, nome)
    assert tensor.shape == atual.shape, nome
    assert torch.isfinite(tensor).all().item(), nome

outnorm3_antes = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in norma.state_dict().items()
}

resultado = norma.load_state_dict(novos_pesos, strict=True)
norma.eval().requires_grad_(False)

for nome, esperado in novos_pesos.items():
    atual = getattr(norma, nome).detach()
    assert torch.equal(
        atual, esperado.to(device=atual.device, dtype=atual.dtype)
    ), f"Cópia divergente: {nome}"

print(resultado)
print("Transferência conferida: classifier.norm → outnorm3")
print("Modelo em eval():", not encoder.training)
print("Encoder congelado:", not any(
    p.requires_grad for p in encoder.parameters()
))

del checkpoint, novos_pesos

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

assert VARIANTE == "base"
assert alpnet.encoder is encoder
assert not alpnet.training and not protosam.training
assert protosam.use_cca is False
assert "outnorm3_antes" in globals()

# Preserva os parâmetros treinados atualmente carregados.
outnorm3_treinada = {
    nome: tensor.detach().cpu().clone()
    for nome, tensor in encoder.outnorm3.state_dict().items()
}

nome_query = Path(ds_teste.images[0]).name
assert nome_query == "cju0u82z3cuma0835wlxrnrjv.png"

item = ds_teste[0]
device = next(encoder.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
qry = item["image"].unsqueeze(0).to(device)
gt = item["label"].cpu().numpy() > 0.5

def nova_entrada():
    return ALPNetInput(
        support_images=[sup_img],
        support_labels=[sup_mask],
        query_images=qry,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )

def metricas_mascara(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "pixels_FG": int(pred.sum()),
    }

mascaras_normalizacao = {}
linhas_normalizacao = []
wrapper_diagnostico = ALPNetWrapper(alpnet)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for condicao, pesos in (
            ("Inicial", outnorm3_antes),
            ("Treinada", outnorm3_treinada),
        ):
            encoder.outnorm3.load_state_dict(pesos, strict=True)

            logits = wrapper_diagnostico(nova_entrada())
            assert torch.isfinite(logits).all().item()
            pred_alp = logits.argmax(dim=1)[0].cpu().numpy() > 0
            del logits

            saida, scores = protosam(
                query_image=qry,
                coarse_model_input=nova_entrada(),
                degrees_rotate=0,
            )
            assert torch.isfinite(saida).all().item()
            assert ((saida == 0) | (saida == 1)).all().item()
            pred_sam = saida.cpu().numpy() > 0

            for etapa, pred in (("ALP", pred_alp), ("SAM", pred_sam)):
                assert pred.shape == gt.shape
                mascaras_normalizacao[(condicao, etapa)] = pred.copy()
                linhas_normalizacao.append({
                    "normalizacao": condicao,
                    "etapa": etapa,
                    **metricas_mascara(pred, gt),
                })

            print(f"{condicao}: concluída | scores SAM: {len(scores)}")

finally:
    encoder.outnorm3.load_state_dict(outnorm3_antes, strict=True)

print("\nQuery:", nome_query)
display(pd.DataFrame(linhas_normalizacao).round(6))
print("Normalização inicial restaurada.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
from PIL import Image
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

REPO = Path("/workspace/ProtoSAM-repro")
REF = REPO / "reproducao" / (
    "vmamba_base_k1_selecao_sam_exploratoria_20261010T180702018291Z"
)

assert VARIANTE == "base"
assert alpnet.encoder is encoder
assert not alpnet.training and not protosam.training
assert not protosam.use_cca
assert "outnorm3_antes" in globals()
assert "outnorm3_treinada" in globals()

referencia = pd.read_csv(REF / "por_imagem.csv")
config_ref = json.loads((REF / "config.json").read_text())

assert len(referencia) == len(ds_teste) == 100
assert referencia["query"].is_unique
assert config_ref["suporte"] == SUPORTE_K1
assert config_ref["gpu"] == torch.cuda.get_device_name()
assert config_ref["torch"] == str(torch.__version__)
assert config_ref["cuda"] == torch.version.cuda
assert config_ref["cudnn"] == torch.backends.cudnn.version()
assert config_ref["tf32_matmul"] == torch.backends.cuda.matmul.allow_tf32
assert config_ref["tf32_cudnn"] == torch.backends.cudnn.allow_tf32

for atributo in ("use_cca", "point_mode", "use_bbox", "use_neg_points"):
    assert config_ref[atributo] == getattr(protosam, atributo), atributo

codigo_selecao = inspect.getsource(type(protosam).predict_w_points_bbox)
assert hashlib.sha256(codigo_selecao.encode()).hexdigest() == (
    config_ref["codigo_selecao_sha256"]
)

for linha in referencia.to_dict("records"):
    i = int(linha["indice"])
    assert Path(ds_teste.images[i]).name == linha["query"]

def salvar_json_diagnostico(caminho, dados):
    tmp = caminho.with_suffix(caminho.suffix + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    tmp.replace(caminho)

def hash_arquivo(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_NORM = REPO / "reproducao" / (
    f"vmamba_base_k1_norm_treinada_{stamp}"
)
DIR_NORM.mkdir(parents=True, exist_ok=False)
for pasta in ("alp", "sam", "gt", "registros"):
    (DIR_NORM / pasta).mkdir()

checkpoint_path = REPO / "pretrained_model/vmamba_base_v2.pth"

# Registra exatamente os parâmetros aplicados.
torch.save({
    "inicial": outnorm3_antes,
    "treinada": outnorm3_treinada,
}, DIR_NORM / "normalizacoes.pt")

salvar_json_diagnostico(DIR_NORM / "config.json", {
    "tipo": "ablacao exploratoria de normalizacao",
    "encoder": "vmamba_base",
    "suporte": SUPORTE_K1,
    "normalizacao": "classifier.norm transferida para outnorm3",
    "selecao_sam": "alternativa_0_por_componente",
    "referencia": str(REF),
    "gpu": torch.cuda.get_device_name(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "config_alp": alpnet.config,
    "checkpoint_sha256": hash_arquivo(checkpoint_path),
    "normalizacoes_sha256": hash_arquivo(DIR_NORM / "normalizacoes.pt"),
    "codigo_vmamba_sha256": hash_arquivo(REPO / "models/vmamba.py"),
    "codigo_protosam_sha256": hash_arquivo(REPO / "models/ProtoSAM.py"),
    "config_referencia": config_ref,
})
referencia.to_csv(DIR_NORM / "referencia_por_imagem.csv", index=False)

device = next(encoder.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
wrapper_norm = ALPNetWrapper(alpnet)

def medir_norm(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "TP": tp, "FP": fp, "FN": fn,
    }

linhas_norm = []
estado, erro = "executando", None
print("Diretório:", DIR_NORM, flush=True)

try:
    encoder.outnorm3.load_state_dict(outnorm3_treinada, strict=True)

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for ref in referencia.sort_values("indice").to_dict("records"):
            nome = ref["query"]
            item = ds_teste[int(ref["indice"])]
            gt = item["label"].cpu().numpy() > 0.5
            qry = item["image"].unsqueeze(0).to(device)

            # Confere a métrica da máscara de referência salva.
            with Image.open(REF / "original" / nome) as imagem:
                sam_ref = np.asarray(imagem.convert("L")) > 0
            assert sam_ref.shape == gt.shape
            dice_ref = medir_norm(sam_ref, gt)["Dice"]
            assert np.isclose(
                dice_ref, ref["Dice_original"], rtol=0, atol=1e-10
            ), f"Referência divergente: {nome}"

            def entrada_norm():
                return ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True, val_wsize=2,
                    show_viz=False, supp_fts=None,
                )

            logits = wrapper_norm(entrada_norm())
            assert torch.isfinite(logits).all().item()
            pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
            del logits

            saida, scores = protosam(
                query_image=qry,
                coarse_model_input=entrada_norm(),
                degrees_rotate=0,
            )
            assert torch.isfinite(saida).all().item()
            assert ((saida == 0) | (saida == 1)).all().item()
            pred_sam = saida.cpu().numpy() > 0

            linha = {
                "indice": int(ref["indice"]),
                "query": nome,
                "SAM_Dice_norm_inicial": dice_ref,
                "componentes": len(scores),
            }
            for etapa, mascara in (("ALP", pred_alp), ("SAM", pred_sam)):
                assert mascara.shape == gt.shape
                linha.update({
                    f"{etapa}_{k}": v
                    for k, v in medir_norm(mascara, gt).items()
                })
                Image.fromarray(
                    mascara.astype(np.uint8) * 255
                ).save(DIR_NORM / etapa.lower() / nome)

            Image.fromarray(gt.astype(np.uint8) * 255).save(
                DIR_NORM / "gt" / nome
            )
            linha["delta_SAM"] = linha["SAM_Dice"] - dice_ref

            salvar_json_diagnostico(
                DIR_NORM / "registros" / f"{Path(nome).stem}.json", linha
            )
            linhas_norm.append(linha)

            tmp = DIR_NORM / "por_imagem.tmp.csv"
            pd.DataFrame(linhas_norm).to_csv(tmp, index=False)
            tmp.replace(DIR_NORM / "por_imagem.csv")

            print(
                f"[{len(linhas_norm)}/100] {nome}: "
                f"ALP={linha['ALP_Dice']:.6f} | "
                f"SAM inicial={dice_ref:.6f} "
                f"treinada={linha['SAM_Dice']:.6f} | "
                f"delta={linha['delta_SAM']:+.6f}",
                flush=True,
            )

    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    encoder.outnorm3.load_state_dict(outnorm3_antes, strict=True)
    salvar_json_diagnostico(DIR_NORM / "status.json", {
        "status": estado,
        "avaliadas": len(linhas_norm),
        "previstas": 100,
        "erro": erro,
    })
    print("Normalização inicial restaurada.")

df_norm = pd.DataFrame(linhas_norm)
resumo_norm = {
    "n": len(df_norm),
    "ALP_Dice_norm_treinada": float(df_norm["ALP_Dice"].mean()),
    "ALP_IoU_norm_treinada": float(df_norm["ALP_IoU"].mean()),
    "SAM_Dice_norm_inicial": float(df_norm["SAM_Dice_norm_inicial"].mean()),
    "SAM_Dice_norm_treinada": float(df_norm["SAM_Dice"].mean()),
    "SAM_IoU_norm_treinada": float(df_norm["SAM_IoU"].mean()),
    "delta_SAM_medio": float(df_norm["delta_SAM"].mean()),
    "melhora": int((df_norm["delta_SAM"] > 0).sum()),
    "piora": int((df_norm["delta_SAM"] < 0).sum()),
    "empate": int((df_norm["delta_SAM"] == 0).sum()),
}
salvar_json_diagnostico(DIR_NORM / "resumo.json", resumo_norm)
print(json.dumps(resumo_norm, indent=2))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

REPO = Path("/workspace/ProtoSAM-repro")
REF_SMALL = REPO / "reproducao" / (
    "vmamba_small_k1_avaliar_20261008T110134736074Z"
)
PESOS_SMALL = REPO / "pretrained_model/vmamba_small_v2.pth"

assert VARIANTE == "small"
assert alpnet.encoder is encoder
assert tuple(encoder.outnorm3.weight.shape) == (768,)
assert not alpnet.training and not protosam.training
assert not protosam.sam.training
assert not any(p.requires_grad for p in encoder.parameters())
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_mask and not protosam.use_neg_points
assert not protosam.coarse_pred_only
assert torch.all(encoder.outnorm3.weight == 1).item()
assert torch.all(encoder.outnorm3.bias == 0).item()

referencia = pd.read_csv(REF_SMALL / "por_imagem.csv")
assert len(referencia) == len(ds_teste) == 100
assert referencia["query"].is_unique
assert referencia["indice"].is_unique

for r in referencia.to_dict("records"):
    assert Path(ds_teste.images[int(r["indice"])]).name == r["query"]

norm_inicial_small = {
    k: v.detach().cpu().clone()
    for k, v in encoder.outnorm3.state_dict().items()
}
checkpoint = torch.load(
    PESOS_SMALL, map_location="cpu", weights_only=True
)
norm_treinada_small = {
    k: checkpoint["model"][f"classifier.norm.{k}"].detach().cpu().clone()
    for k in ("weight", "bias")
}
del checkpoint

for k in norm_inicial_small:
    assert norm_treinada_small[k].shape == norm_inicial_small[k].shape
    assert torch.isfinite(norm_treinada_small[k]).all().item()

def hash_small(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def salvar_small(path, dados):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    tmp.replace(path)

def medir_small(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "TP": tp, "FP": fp, "FN": fn,
    }

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_SMALL_NORM = REPO / "reproducao" / (
    f"vmamba_small_k1_norm_pareada_{stamp}"
)
DIR_SMALL_NORM.mkdir(parents=True, exist_ok=False)

for pasta in (
    "inicial/alp", "inicial/sam",
    "treinada/alp", "treinada/sam", "gt", "registros",
):
    (DIR_SMALL_NORM / pasta).mkdir(parents=True)

torch.save(
    {"inicial": norm_inicial_small, "treinada": norm_treinada_small},
    DIR_SMALL_NORM / "normalizacoes.pt",
)
referencia.to_csv(DIR_SMALL_NORM / "referencia_4090.csv", index=False)

salvar_small(DIR_SMALL_NORM / "config.json", {
    "tipo": "ablacao exploratoria pareada de normalizacao",
    "encoder": "vmamba_small",
    "suporte": SUPORTE_K1,
    "condicoes": ["inicial", "classifier.norm transferida para outnorm3"],
    "selecao_sam": "alternativa_0_por_componente",
    "referencia_4090": str(REF_SMALL),
    "gpu_driver_memoria": subprocess.check_output([
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ], text=True).strip(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "config_alp": alpnet.config,
    "resolucao_encoder": alpnet.image_size,
    "point_mode": protosam.point_mode,
    "use_cca": protosam.use_cca,
    "checkpoint_sha256": hash_small(PESOS_SMALL),
    "normalizacoes_sha256": hash_small(DIR_SMALL_NORM / "normalizacoes.pt"),
    "vmamba_sha256": hash_small(REPO / "models/vmamba.py"),
    "protosam_sha256": hash_small(REPO / "models/ProtoSAM.py"),
})

device = next(encoder.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
wrapper_small = ALPNetWrapper(alpnet)

linhas_small = []
estado, erro = "executando", None
print("Diretório:", DIR_SMALL_NORM, flush=True)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for ref in referencia.sort_values("indice").to_dict("records"):
            nome = ref["query"]
            item = ds_teste[int(ref["indice"])]
            gt = item["label"].cpu().numpy() > 0.5
            qry = item["image"].unsqueeze(0).to(device)

            with Image.open(REF_SMALL / "alp" / nome) as img:
                alp_ref = np.asarray(img.convert("L")) > 0
            assert alp_ref.shape == gt.shape
            mref = medir_small(alp_ref, gt)
            for k in ("TP", "FP", "FN"):
                assert mref[k] == int(ref[f"ALP_{k}"]), (nome, k)

            linha = {"indice": int(ref["indice"]), "query": nome}

            def entrada_small():
                return ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True, val_wsize=2,
                    show_viz=False, supp_fts=None,
                )

            for condicao, pesos in (
                ("inicial", norm_inicial_small),
                ("treinada", norm_treinada_small),
            ):
                encoder.outnorm3.load_state_dict(pesos, strict=True)

                logits = wrapper_small(entrada_small())
                assert torch.isfinite(logits).all().item()
                pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
                del logits

                saida, scores = protosam(
                    query_image=qry,
                    coarse_model_input=entrada_small(),
                    degrees_rotate=0,
                )
                assert torch.isfinite(saida).all().item()
                assert ((saida == 0) | (saida == 1)).all().item()
                pred_sam = saida.cpu().numpy() > 0

                linha[f"{condicao}_componentes"] = len(scores)
                for etapa, mascara in (("ALP", pred_alp), ("SAM", pred_sam)):
                    assert mascara.shape == gt.shape
                    linha.update({
                        f"{condicao}_{etapa}_{k}": v
                        for k, v in medir_small(mascara, gt).items()
                    })
                    Image.fromarray(
                        mascara.astype(np.uint8) * 255
                    ).save(DIR_SMALL_NORM / condicao / etapa.lower() / nome)

            Image.fromarray(gt.astype(np.uint8) * 255).save(
                DIR_SMALL_NORM / "gt" / nome
            )
            for etapa in ("ALP", "SAM"):
                linha[f"delta_{etapa}"] = (
                    linha[f"treinada_{etapa}_Dice"]
                    - linha[f"inicial_{etapa}_Dice"]
                )

            salvar_small(
                DIR_SMALL_NORM / "registros" / f"{Path(nome).stem}.json",
                linha,
            )
            linhas_small.append(linha)
            tmp = DIR_SMALL_NORM / "por_imagem.tmp.csv"
            pd.DataFrame(linhas_small).to_csv(tmp, index=False)
            tmp.replace(DIR_SMALL_NORM / "por_imagem.csv")

            print(
                f"[{len(linhas_small)}/100] {nome}: "
                f"ALP {linha['inicial_ALP_Dice']:.6f}"
                f" → {linha['treinada_ALP_Dice']:.6f} | "
                f"SAM {linha['inicial_SAM_Dice']:.6f}"
                f" → {linha['treinada_SAM_Dice']:.6f}",
                flush=True,
            )

    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    encoder.outnorm3.load_state_dict(norm_inicial_small, strict=True)
    salvar_small(DIR_SMALL_NORM / "status.json", {
        "status": estado,
        "avaliadas": len(linhas_small),
        "previstas": 100,
        "erro": erro,
    })
    print("Normalização inicial restaurada.")

df_small = pd.DataFrame(linhas_small)
resumo_small = {"n": len(df_small)}

for etapa in ("ALP", "SAM"):
    delta = df_small[f"delta_{etapa}"]
    resumo_small[etapa] = {
        "Dice_inicial": float(df_small[f"inicial_{etapa}_Dice"].mean()),
        "Dice_treinada": float(df_small[f"treinada_{etapa}_Dice"].mean()),
        "IoU_inicial": float(df_small[f"inicial_{etapa}_IoU"].mean()),
        "IoU_treinada": float(df_small[f"treinada_{etapa}_IoU"].mean()),
        "delta_medio": float(delta.mean()),
        "melhora": int((delta > 0).sum()),
        "piora": int((delta < 0).sum()),
        "empate": int((delta == 0).sum()),
    }

salvar_small(DIR_SMALL_NORM / "resumo.json", resumo_small)
print(json.dumps(resumo_small, indent=2))

In [ ]:
from pathlib import Path
import sys
import subprocess
import torch

REPO = Path("/workspace/ProtoSAM-repro")
ULTIMA_EXECUCAO = REPO / "reproducao" / (
    "vmamba_small_k1_norm_pareada_20261010T210704936556Z"
)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("CUDA disponível:", torch.cuda.is_available())

resultado = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)
print(resultado.stdout or resultado.stderr)

arquivos = {
    "Código VMamba": REPO / "models/vmamba.py",
    "Código ALP": REPO / "models/grid_proto_fewshot.py",
    "Código ProtoSAM": REPO / "models/ProtoSAM.py",
    "Checkpoint Small": REPO / "pretrained_model/vmamba_small_v2.pth",
    "Checkpoint SAM-H": REPO / "pretrained_model/sam_vit_h.pth",
    "Resumo da última avaliação": ULTIMA_EXECUCAO / "resumo.json",
    "Resultados por imagem": ULTIMA_EXECUCAO / "por_imagem.csv",
}

print("\n=== Arquivos ===")
for descricao, caminho in arquivos.items():
    print(f"{descricao}: {'OK' if caminho.is_file() else 'AUSENTE'}")

dados = REPO / "data/PolypDataset"
print("Dataset:", "OK" if dados.is_dir() else "AUSENTE")

In [ ]:
import inspect
import torch

assert VARIANTE == "small"
assert alpnet.encoder is encoder
assert not alpnet.training and not protosam.training

print("GPU:", torch.cuda.get_device_name())
print("Dimensões por estágio:", encoder.dims)
print("Estágios retornados:", encoder.out_indices)
print("Resolução de entrada:", alpnet.image_size)
print("Queries:", len(ds_teste))
print("Suporte:", SUPORTE_K1)
print(
    "Normalização inicial:",
    bool((encoder.outnorm3.weight == 1).all().item())
    and bool((encoder.outnorm3.bias == 0).all().item()),
)

print("\n=== Extração de características ===")
print(inspect.getsource(type(alpnet).get_features))

print("\n=== Construção do classificador ALP ===")
print(inspect.getsource(type(alpnet).get_cls))

In [ ]:
import torch
import torch.nn.functional as F

assert VARIANTE == "small"
assert encoder.out_indices == (3,)
assert encoder.channel_first
assert not encoder.training

captura_estagio2 = {}

def capturar_estagio2(modulo, entradas, saida):
    captura_estagio2["features"] = saida.detach()

# Índice 2 = terceiro estágio, antes do seu downsample.
hook = encoder.layers[2].blocks.register_forward_hook(
    capturar_estagio2
)

device = next(encoder.parameters()).device
item = ds_teste[0]
entrada = item["image"].unsqueeze(0).to(device)
entrada = F.interpolate(
    entrada,
    size=(672, 672),
    mode="bilinear",
    align_corners=False,
)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        saidas = encoder(entrada)
        features = captura_estagio2["features"]

        print("Entrada:", tuple(entrada.shape))
        print("Penúltimo estágio:", tuple(features.shape))
        print("Último estágio:", tuple(saidas[-1].shape))
        print("Dtype:", features.dtype)
        print("Valores finitos:", torch.isfinite(features).all().item())

        assert tuple(features.shape) == (1, 384, 42, 42)
        assert torch.isfinite(features).all().item()
finally:
    hook.remove()
    captura_estagio2.clear()

del entrada, saidas, features

In [ ]:
from copy import deepcopy
import torch
from torch import nn
import torch.nn.functional as F
import models.grid_proto_fewshot as modulo_alp

assert VARIANTE == "small"
assert encoder.channel_first
assert getattr(encoder, "pos_embed", None) is None
assert "alpnet_estagio2" not in globals(), (
    "A variante já foi criada neste kernel."
)

class EncoderSmallEstagio2(nn.Module):
    def __init__(self, origem):
        super().__init__()
        self.patch_embed = origem.patch_embed

        # Estágios 0 e 1 completos, incluindo downsample.
        self.estagios_iniciais = nn.ModuleList([
            origem.layers[0],
            origem.layers[1],
        ])

        # Estágio 2: somente os blocos, antes do downsample.
        self.blocos_estagio2 = origem.layers[2].blocks

        # Mesma implementação de LayerNorm, agora com 384 canais.
        self.outnorm2 = type(origem.outnorm3)(
            384,
            eps=origem.outnorm3.eps,
            channel_first=True,
        )

    def forward(self, x):
        x = self.patch_embed(x)
        for estagio in self.estagios_iniciais:
            x = estagio(x)
        x = self.blocos_estagio2(x)
        return self.outnorm2(x).contiguous()


device = next(encoder.parameters()).device
encoder_estagio2 = EncoderSmallEstagio2(encoder)
encoder_estagio2.to(device=device, dtype=torch.float32)
encoder_estagio2.eval().requires_grad_(False)

class FewShotSegSmallEstagio2(modulo_alp.FewShotSeg):
    def get_encoder(self):
        self.encoder = encoder_estagio2
        self.config["feature_hw"] = [42, 42]

    def get_cls(self):
        assert self.config["cls_name"] == "grid_proto"
        grade = self.config["proto_grid_size"]
        self.cls_unit = modulo_alp.MultiProtoAsConv(
            proto_grid=[grade, grade],
            feature_hw=self.config["feature_hw"],
            embed_dim=384,
        )

    def get_features(self, imgs_concat):
        imgs_concat = F.interpolate(
            imgs_concat,
            size=(672, 672),
            mode="bilinear",
            align_corners=False,
        )
        features = self.encoder(imgs_concat)
        assert tuple(features.shape[1:]) == (384, 42, 42)
        return features


alpnet_estagio2 = FewShotSegSmallEstagio2(
    image_size=672,
    pretrained_path=None,
    cfg=deepcopy(alpnet.config),
).to(device=device, dtype=torch.float32)

alpnet_estagio2.eval().requires_grad_(False)

assert type(alpnet_estagio2).forward is modulo_alp.FewShotSeg.forward
assert not any(p.requires_grad for p in alpnet_estagio2.parameters())

# Confere se o encoder truncado reproduz os blocos do modelo completo.
captura = {}

def capturar(modulo, entradas, saida):
    captura["features"] = saida.detach()

hook = encoder.layers[2].blocks.register_forward_hook(capturar)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        entrada = F.interpolate(
            ds_teste[0]["image"].unsqueeze(0).to(device),
            size=(672, 672),
            mode="bilinear",
            align_corners=False,
        )

        saidas_completas = encoder(entrada)
        esperado = encoder_estagio2.outnorm2(
            captura["features"]
        ).clone()

        obtido = encoder_estagio2(entrada)

        assert torch.isfinite(obtido).all().item()
        torch.testing.assert_close(
            obtido, esperado, rtol=1e-5, atol=1e-6
        )

        print("Saída:", tuple(obtido.shape))
        print(
            "Diferença absoluta máxima:",
            (obtido - esperado).abs().max().item(),
        )
finally:
    hook.remove()
    captura.clear()

print("Forward oficial do ALP preservado.")
print("Grade ALP:", alpnet_estagio2.config["feature_hw"])
print("Normalização: escala 1 e viés 0, congelados.")
print("Último estágio não faz parte do novo encoder.")

del entrada, saidas_completas, esperado, obtido

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

assert not alpnet_estagio2.training and not protosam.training
assert not protosam.use_cca
assert protosam.point_mode == "both"

nome_query_estagio2 = Path(ds_teste.images[0]).name
assert nome_query_estagio2 == "cju0u82z3cuma0835wlxrnrjv.png"

device = next(alpnet_estagio2.parameters()).device
item_estagio2 = ds_teste[0]
gt_estagio2 = item_estagio2["label"].cpu().numpy() > 0.5

sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
qry_estagio2 = item_estagio2["image"].unsqueeze(0).to(device)

def entrada_estagio2():
    return ALPNetInput(
        support_images=[sup_img],
        support_labels=[sup_mask],
        query_images=qry_estagio2,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )

def medir_estagio2(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "precisao": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "pixels_FG": int(pred.sum()),
    }

wrapper_estagio2 = ALPNetWrapper(alpnet_estagio2)
modelo_anterior = protosam.coarse_segmentation_model

try:
    protosam.coarse_segmentation_model = wrapper_estagio2

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        logits = wrapper_estagio2(entrada_estagio2())
        assert tuple(logits.shape) == (1, 2, 1024, 1024)
        assert torch.isfinite(logits).all().item()
        pred_alp_estagio2 = logits.argmax(1)[0].cpu().numpy() > 0
        del logits

        saida, scores_estagio2 = protosam(
            query_image=qry_estagio2,
            coarse_model_input=entrada_estagio2(),
            degrees_rotate=0,
        )
        assert tuple(saida.shape) == (1024, 1024)
        assert torch.isfinite(saida).all().item()
        assert ((saida == 0) | (saida == 1)).all().item()
        pred_sam_estagio2 = saida.cpu().numpy() > 0

finally:
    protosam.coarse_segmentation_model = modelo_anterior

print("Query:", nome_query_estagio2)
print("Quantidade de scores SAM:", len(scores_estagio2))
display(pd.DataFrame({
    "ALP — penúltimo estágio": medir_estagio2(
        pred_alp_estagio2, gt_estagio2
    ),
    "SAM — penúltimo estágio": medir_estagio2(
        pred_sam_estagio2, gt_estagio2
    ),
}))
print("Modelo original reconectado ao ProtoSAM.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO = Path("/workspace/ProtoSAM-repro")
REF = REPO / "reproducao" / (
    "vmamba_small_k1_norm_pareada_20261010T210704936556Z"
)

assert VARIANTE == "small"
assert not alpnet_estagio2.training and not protosam.training
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_neg_points and not protosam.use_mask
assert not protosam.coarse_pred_only

referencia = pd.read_csv(REF / "por_imagem.csv")
assert len(referencia) == len(ds_teste) == 100
assert referencia["query"].is_unique

for r in referencia.to_dict("records"):
    assert Path(ds_teste.images[int(r["indice"])]).name == r["query"]

def salvar_e2(path, dados):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    tmp.replace(path)

def hash_e2(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_ESTAGIO2 = REPO / "reproducao" / (
    f"vmamba_small_k1_estagio2_grade42_{stamp}"
)
DIR_ESTAGIO2.mkdir(parents=True, exist_ok=False)
for pasta in ("alp", "sam", "gt", "registros"):
    (DIR_ESTAGIO2 / pasta).mkdir()

referencia.to_csv(DIR_ESTAGIO2 / "referencia.csv", index=False)
salvar_e2(DIR_ESTAGIO2 / "config.json", {
    "tipo": "avaliacao exploratoria de estagio",
    "encoder": "vmamba_small",
    "estagio_indice": 2,
    "canais": 384,
    "grade": [42, 42],
    "resolucao_encoder": 672,
    "normalizacao": "LayerNorm com escala 1 e vies 0",
    "selecao_sam": "alternativa_0_por_componente",
    "suporte": SUPORTE_K1,
    "config_alp": alpnet_estagio2.config,
    "referencia_anterior": str(REF),
    "gpu_driver": subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,driver_version",
        "--format=csv,noheader",
    ], text=True).strip(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "checkpoint_sha256": hash_e2(
        REPO / "pretrained_model/vmamba_small_v2.pth"
    ),
    "vmamba_sha256": hash_e2(REPO / "models/vmamba.py"),
    "protosam_sha256": hash_e2(REPO / "models/ProtoSAM.py"),
})

device = next(alpnet_estagio2.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)

linhas_e2 = []
estado, erro = "executando", None
modelo_anterior = protosam.coarse_segmentation_model
print("Diretório:", DIR_ESTAGIO2, flush=True)

try:
    protosam.coarse_segmentation_model = wrapper_estagio2

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for r in referencia.sort_values("indice").to_dict("records"):
            nome = r["query"]
            item = ds_teste[int(r["indice"])]
            gt = item["label"].cpu().numpy() > 0.5

            with Image.open(REF / "gt" / nome) as img:
                gt_ref = np.asarray(img.convert("L")) > 0
            assert np.array_equal(gt, gt_ref), nome

            qry = item["image"].unsqueeze(0).to(device)

            def entrada_e2():
                return ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True, val_wsize=2,
                    show_viz=False, supp_fts=None,
                )

            logits = wrapper_estagio2(entrada_e2())
            assert tuple(logits.shape) == (1, 2, 1024, 1024)
            assert torch.isfinite(logits).all().item()
            pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
            del logits

            saida, scores = protosam(
                query_image=qry,
                coarse_model_input=entrada_e2(),
                degrees_rotate=0,
            )
            assert torch.isfinite(saida).all().item()
            assert ((saida == 0) | (saida == 1)).all().item()
            pred_sam = saida.cpu().numpy() > 0

            linha = {
                "indice": int(r["indice"]),
                "query": nome,
                "numero_scores_sam": len(scores),
            }
            for etapa, pred in (("ALP", pred_alp), ("SAM", pred_sam)):
                assert pred.shape == gt.shape
                linha.update({
                    f"{etapa}_{k}": v
                    for k, v in medir_estagio2(pred, gt).items()
                })
                Image.fromarray(pred.astype(np.uint8) * 255).save(
                    DIR_ESTAGIO2 / etapa.lower() / nome
                )

            Image.fromarray(gt.astype(np.uint8) * 255).save(
                DIR_ESTAGIO2 / "gt" / nome
            )
            salvar_e2(
                DIR_ESTAGIO2 / "registros" / f"{Path(nome).stem}.json",
                linha,
            )
            linhas_e2.append(linha)
            tmp = DIR_ESTAGIO2 / "por_imagem.tmp.csv"
            pd.DataFrame(linhas_e2).to_csv(tmp, index=False)
            tmp.replace(DIR_ESTAGIO2 / "por_imagem.csv")

            print(
                f"[{len(linhas_e2)}/100] {nome}: "
                f"ALP={linha['ALP_Dice']:.6f} "
                f"SAM={linha['SAM_Dice']:.6f}",
                flush=True,
            )
    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    protosam.coarse_segmentation_model = modelo_anterior
    salvar_e2(DIR_ESTAGIO2 / "status.json", {
        "status": estado,
        "avaliadas": len(linhas_e2),
        "previstas": 100,
        "erro": erro,
    })
    print("Modelo original reconectado.")

df_e2 = pd.DataFrame(linhas_e2)
resumo_e2 = {
    "n": len(df_e2),
    "medias": {
        etapa: {
            m: float(df_e2[f"{etapa}_{m}"].mean())
            for m in ("Dice", "IoU", "precisao", "recall")
        }
        for etapa in ("ALP", "SAM")
    },
}
salvar_e2(DIR_ESTAGIO2 / "resumo.json", resumo_e2)
print(json.dumps(resumo_e2, indent=2))

In [ ]:
from copy import deepcopy
import torch
from torch import nn
import torch.nn.functional as F
import models.grid_proto_fewshot as modulo_alp

assert VARIANTE == "base"
assert alpnet.encoder is encoder
assert list(encoder.dims) == [128, 256, 512, 1024]
assert [len(e.blocks) for e in encoder.layers] == [2, 2, 15, 2]
assert encoder.channel_first
assert getattr(encoder, "pos_embed", None) is None
assert not encoder.training
assert "alpnet_estagio2" not in globals(), (
    "A variante já foi criada neste kernel."
)

class EncoderBaseEstagio2(nn.Module):
    def __init__(self, origem):
        super().__init__()
        self.patch_embed = origem.patch_embed
        self.estagios_iniciais = nn.ModuleList([
            origem.layers[0],
            origem.layers[1],
        ])
        self.blocos_estagio2 = origem.layers[2].blocks
        self.outnorm2 = type(origem.outnorm3)(
            512,
            eps=origem.outnorm3.eps,
            channel_first=True,
        )

    def forward(self, x):
        x = self.patch_embed(x)
        for estagio in self.estagios_iniciais:
            x = estagio(x)
        x = self.blocos_estagio2(x)
        return self.outnorm2(x).contiguous()


device = next(encoder.parameters()).device
encoder_estagio2 = EncoderBaseEstagio2(encoder)
encoder_estagio2.to(device=device, dtype=torch.float32)
encoder_estagio2.eval().requires_grad_(False)

class FewShotSegBaseEstagio2(modulo_alp.FewShotSeg):
    def get_encoder(self):
        self.encoder = encoder_estagio2
        self.config["feature_hw"] = [42, 42]

    def get_cls(self):
        assert self.config["cls_name"] == "grid_proto"
        grade = self.config["proto_grid_size"]
        self.cls_unit = modulo_alp.MultiProtoAsConv(
            proto_grid=[grade, grade],
            feature_hw=self.config["feature_hw"],
            embed_dim=512,
        )

    def get_features(self, imgs_concat):
        imgs_concat = F.interpolate(
            imgs_concat,
            size=(672, 672),
            mode="bilinear",
            align_corners=False,
        )
        features = self.encoder(imgs_concat)
        assert tuple(features.shape[1:]) == (512, 42, 42)
        return features


assert alpnet.config["proto_grid_size"] == 8

alpnet_estagio2 = FewShotSegBaseEstagio2(
    image_size=672,
    pretrained_path=None,
    cfg=deepcopy(alpnet.config),
).to(device=device, dtype=torch.float32)

alpnet_estagio2.eval().requires_grad_(False)

assert type(alpnet_estagio2).forward is modulo_alp.FewShotSeg.forward
assert not any(p.requires_grad for p in alpnet_estagio2.parameters())

# Compara com a saída dos mesmos blocos no encoder completo.
captura = {}

def capturar(modulo, entradas, saida):
    captura["features"] = saida.detach()

hook = encoder.layers[2].blocks.register_forward_hook(capturar)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        entrada = F.interpolate(
            ds_teste[0]["image"].unsqueeze(0).to(device),
            size=(672, 672),
            mode="bilinear",
            align_corners=False,
        )

        saidas_completas = encoder(entrada)
        esperado = encoder_estagio2.outnorm2(
            captura["features"]
        ).clone()
        obtido = encoder_estagio2(entrada)

        assert tuple(obtido.shape) == (1, 512, 42, 42)
        assert torch.isfinite(obtido).all().item()
        torch.testing.assert_close(
            obtido, esperado, rtol=1e-5, atol=1e-6
        )

        print("Saída:", tuple(obtido.shape))
        print(
            "Diferença absoluta máxima:",
            (obtido - esperado).abs().max().item(),
        )
finally:
    hook.remove()
    captura.clear()

print("Forward oficial do ALP preservado.")
print("Grade ALP:", alpnet_estagio2.config["feature_hw"])
print("Normalização: escala 1 e viés 0, congelados.")
print("Último estágio excluído do novo encoder.")

del entrada, saidas_completas, esperado, obtido

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

assert VARIANTE == "base"
assert not alpnet_estagio2.training and not protosam.training
assert not protosam.sam.training
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_neg_points and not protosam.use_mask
assert not protosam.coarse_pred_only

nome_query_estagio2 = Path(ds_teste.images[0]).name
assert nome_query_estagio2 == "cju0u82z3cuma0835wlxrnrjv.png"

device = next(alpnet_estagio2.parameters()).device
item = ds_teste[0]
gt_estagio2 = item["label"].cpu().numpy() > 0.5

sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
qry_estagio2 = item["image"].unsqueeze(0).to(device)

def entrada_estagio2():
    return ALPNetInput(
        support_images=[sup_img],
        support_labels=[sup_mask],
        query_images=qry_estagio2,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )

def medir_estagio2(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "precisao": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "pixels_FG": int(pred.sum()),
    }

wrapper_estagio2 = ALPNetWrapper(alpnet_estagio2)
modelo_anterior = protosam.coarse_segmentation_model

try:
    protosam.coarse_segmentation_model = wrapper_estagio2

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        logits = wrapper_estagio2(entrada_estagio2())
        assert tuple(logits.shape) == (1, 2, 1024, 1024)
        assert torch.isfinite(logits).all().item()
        pred_alp_estagio2 = logits.argmax(1)[0].cpu().numpy() > 0
        del logits

        saida, scores_estagio2 = protosam(
            query_image=qry_estagio2,
            coarse_model_input=entrada_estagio2(),
            degrees_rotate=0,
        )
        assert tuple(saida.shape) == (1024, 1024)
        assert torch.isfinite(saida).all().item()
        assert ((saida == 0) | (saida == 1)).all().item()
        pred_sam_estagio2 = saida.cpu().numpy() > 0

finally:
    protosam.coarse_segmentation_model = modelo_anterior

print("Query:", nome_query_estagio2)
print("Quantidade de scores SAM:", len(scores_estagio2))
display(pd.DataFrame({
    "ALP — Base penúltimo estágio": medir_estagio2(
        pred_alp_estagio2, gt_estagio2
    ),
    "SAM — Base penúltimo estágio": medir_estagio2(
        pred_sam_estagio2, gt_estagio2
    ),
}))
print("Modelo original reconectado ao ProtoSAM.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import json
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO = Path("/workspace/ProtoSAM-repro")
REF = REPO / "reproducao" / (
    "vmamba_base_k1_norm_treinada_20261010T195514199066Z"
)

assert VARIANTE == "base"
assert list(encoder.dims) == [128, 256, 512, 1024]
assert not alpnet_estagio2.training and not protosam.training
assert not protosam.sam.training
assert not any(p.requires_grad for p in alpnet_estagio2.parameters())
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_neg_points and not protosam.use_mask
assert not protosam.coarse_pred_only
assert alpnet_estagio2.config["feature_hw"] == [42, 42]
assert tuple(encoder_estagio2.outnorm2.weight.shape) == (512,)
assert torch.all(encoder_estagio2.outnorm2.weight == 1).item()
assert torch.all(encoder_estagio2.outnorm2.bias == 0).item()

referencia = pd.read_csv(REF / "por_imagem.csv")
assert len(referencia) == len(ds_teste) == 100
assert referencia["query"].is_unique
assert referencia["indice"].is_unique

for r in referencia.to_dict("records"):
    assert Path(ds_teste.images[int(r["indice"])]).name == r["query"]

# Recupera as definições das classes no histórico do Jupyter.
nomes_classes = [
    type(encoder_estagio2).__name__,
    type(alpnet_estagio2).__name__,
]
fontes_classes = {}

for fonte in get_ipython().history_manager.input_hist_raw:
    try:
        arvore = ast.parse(fonte)
    except SyntaxError:
        continue
    for no in arvore.body:
        if isinstance(no, ast.ClassDef) and no.name in nomes_classes:
            fontes_classes[no.name] = ast.get_source_segment(fonte, no)

assert set(nomes_classes) == set(fontes_classes), (
    "Definições das classes não encontradas no histórico."
)

def salvar_e2(path, dados):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    tmp.replace(path)

def hash_e2(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_BASE_ESTAGIO2 = REPO / "reproducao" / (
    f"vmamba_base_k1_estagio2_grade42_{stamp}"
)
DIR_BASE_ESTAGIO2.mkdir(parents=True, exist_ok=False)

for pasta in ("alp", "sam", "gt", "registros"):
    (DIR_BASE_ESTAGIO2 / pasta).mkdir()

referencia.to_csv(
    DIR_BASE_ESTAGIO2 / "referencia.csv", index=False
)
(DIR_BASE_ESTAGIO2 / "variante.py").write_text(
    "\n\n".join(fontes_classes[n] for n in nomes_classes) + "\n",
    encoding="utf-8",
)

salvar_e2(DIR_BASE_ESTAGIO2 / "config.json", {
    "tipo": "avaliacao exploratoria de estagio",
    "encoder": "vmamba_base",
    "estagio_indice": 2,
    "canais": 512,
    "grade": [42, 42],
    "resolucao_encoder": 672,
    "normalizacao": "LayerNorm com escala 1 e vies 0",
    "selecao_sam": "alternativa_0_por_componente",
    "suporte": SUPORTE_K1,
    "config_alp": alpnet_estagio2.config,
    "referencia_anterior": str(REF),
    "gpu_driver": subprocess.check_output([
        "nvidia-smi",
        "--query-gpu=name,driver_version",
        "--format=csv,noheader",
    ], text=True).strip(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "checkpoint_sha256": hash_e2(
        REPO / "pretrained_model/vmamba_base_v2.pth"
    ),
    "vmamba_sha256": hash_e2(REPO / "models/vmamba.py"),
    "protosam_sha256": hash_e2(REPO / "models/ProtoSAM.py"),
})

device = next(alpnet_estagio2.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)

linhas_e2 = []
estado, erro = "executando", None
modelo_anterior = protosam.coarse_segmentation_model
print("Diretório:", DIR_BASE_ESTAGIO2, flush=True)

try:
    protosam.coarse_segmentation_model = wrapper_estagio2

    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for r in referencia.sort_values("indice").to_dict("records"):
            nome = r["query"]
            item = ds_teste[int(r["indice"])]
            gt = item["label"].cpu().numpy() > 0.5

            with Image.open(REF / "gt" / nome) as img:
                gt_ref = np.asarray(img.convert("L")) > 0
            assert np.array_equal(gt, gt_ref), nome

            qry = item["image"].unsqueeze(0).to(device)

            def entrada_e2():
                return ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True,
                    val_wsize=2,
                    show_viz=False,
                    supp_fts=None,
                )

            logits = wrapper_estagio2(entrada_e2())
            assert tuple(logits.shape) == (1, 2, 1024, 1024)
            assert torch.isfinite(logits).all().item()
            pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
            del logits

            saida, scores = protosam(
                query_image=qry,
                coarse_model_input=entrada_e2(),
                degrees_rotate=0,
            )
            assert torch.isfinite(saida).all().item()
            assert ((saida == 0) | (saida == 1)).all().item()
            pred_sam = saida.cpu().numpy() > 0

            linha = {
                "indice": int(r["indice"]),
                "query": nome,
                "numero_scores_sam": len(scores),
            }
            for etapa, pred in (("ALP", pred_alp), ("SAM", pred_sam)):
                assert pred.shape == gt.shape
                linha.update({
                    f"{etapa}_{k}": v
                    for k, v in medir_estagio2(pred, gt).items()
                })
                Image.fromarray(pred.astype(np.uint8) * 255).save(
                    DIR_BASE_ESTAGIO2 / etapa.lower() / nome
                )

            Image.fromarray(gt.astype(np.uint8) * 255).save(
                DIR_BASE_ESTAGIO2 / "gt" / nome
            )
            salvar_e2(
                DIR_BASE_ESTAGIO2 / "registros" / f"{Path(nome).stem}.json",
                linha,
            )
            linhas_e2.append(linha)

            tmp = DIR_BASE_ESTAGIO2 / "por_imagem.tmp.csv"
            pd.DataFrame(linhas_e2).to_csv(tmp, index=False)
            tmp.replace(DIR_BASE_ESTAGIO2 / "por_imagem.csv")

            print(
                f"[{len(linhas_e2)}/100] {nome}: "
                f"ALP={linha['ALP_Dice']:.6f} "
                f"SAM={linha['SAM_Dice']:.6f}",
                flush=True,
            )

    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    protosam.coarse_segmentation_model = modelo_anterior
    salvar_e2(DIR_BASE_ESTAGIO2 / "status.json", {
        "status": estado,
        "avaliadas": len(linhas_e2),
        "previstas": 100,
        "erro": erro,
    })
    print("Modelo original reconectado.")

df_e2 = pd.DataFrame(linhas_e2)
resumo_e2 = {
    "n": len(df_e2),
    "medias": {
        etapa: {
            m: float(df_e2[f"{etapa}_{m}"].mean())
            for m in ("Dice", "IoU", "precisao", "recall")
        }
        for etapa in ("ALP", "SAM")
    },
}
salvar_e2(DIR_BASE_ESTAGIO2 / "resumo.json", resumo_e2)
print(json.dumps(resumo_e2, indent=2))

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F

assert VARIANTE == "base"
assert list(encoder.dims) == [128, 256, 512, 1024]
assert encoder.channel_first
assert getattr(encoder, "pos_embed", None) is None
assert not encoder.training
assert "encoders_fusao" not in globals(), (
    "As fusões já foram criadas neste kernel."
)

class EncoderVMambaFusao(nn.Module):
    def __init__(self, origem, estagios):
        super().__init__()
        self.estagios = tuple(estagios)
        self.ultimo = max(self.estagios)
        self.grade = (42, 42)

        self.patch_embed = origem.patch_embed
        self.blocos = nn.ModuleList([
            origem.layers[i].blocks
            for i in range(self.ultimo + 1)
        ])
        # Não inclui downsample depois do último estágio utilizado.
        self.downsamples = nn.ModuleList([
            origem.layers[i].downsample
            for i in range(self.ultimo)
        ])

        self.normas = nn.ModuleDict({
            str(i): type(origem.outnorm3)(
                origem.dims[i],
                eps=origem.outnorm3.eps,
                channel_first=True,
            )
            for i in self.estagios
        })
        self.canais_saida = sum(origem.dims[i] for i in self.estagios)

    def extrair_mapas(self, x):
        x = self.patch_embed(x)
        mapas = {}

        for i, blocos in enumerate(self.blocos):
            x = blocos(x)
            if i in self.estagios:
                mapas[i] = x
            if i < self.ultimo:
                x = self.downsamples[i](x)

        return mapas

    def combinar(self, mapas):
        partes = []

        for i in self.estagios:
            x = self.normas[str(i)](mapas[i])

            if x.shape[-2:] != self.grade:
                if all(a >= b for a, b in zip(x.shape[-2:], self.grade)):
                    x = F.adaptive_avg_pool2d(x, self.grade)
                else:
                    x = F.interpolate(
                        x, size=self.grade,
                        mode="bilinear", align_corners=False,
                    )

            # Mesma norma por posição para cada estágio.
            x = F.normalize(x, p=2, dim=1, eps=1e-6)
            partes.append(x)

        return torch.cat(partes, dim=1) / math.sqrt(len(partes))

    def forward(self, x):
        return self.combinar(self.extrair_mapas(x))


device = next(encoder.parameters()).device
encoders_fusao = {
    "estagios_1_2": EncoderVMambaFusao(encoder, (1, 2)),
    "estagios_2_3": EncoderVMambaFusao(encoder, (2, 3)),
}

for modelo in encoders_fusao.values():
    modelo.to(device=device, dtype=torch.float32)
    modelo.eval().requires_grad_(False)

# Extrai os mapas do modelo completo para conferir equivalência.
capturas = {}
hooks = []

def criar_hook(indice):
    def capturar(modulo, entradas, saida):
        capturas[indice] = saida.detach()
    return capturar

for i in (1, 2, 3):
    hooks.append(
        encoder.layers[i].blocks.register_forward_hook(criar_hook(i))
    )

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        entrada = F.interpolate(
            ds_teste[0]["image"].unsqueeze(0).to(device),
            size=(672, 672),
            mode="bilinear", align_corners=False,
        )
        saidas_completas = encoder(entrada)
finally:
    for hook in hooks:
        hook.remove()

with torch.inference_mode(), torch.autocast("cuda", enabled=False):
    for nome, modelo in encoders_fusao.items():
        esperado = modelo.combinar(capturas)
        obtido = modelo(entrada)

        assert tuple(obtido.shape) == (
            1, modelo.canais_saida, 42, 42
        )
        assert torch.isfinite(obtido).all().item()
        torch.testing.assert_close(
            obtido, esperado, rtol=1e-5, atol=1e-6
        )

        print(
            nome,
            "| saída:", tuple(obtido.shape),
            "| diferença máxima:",
            (obtido - esperado).abs().max().item(),
        )

capturas.clear()
del entrada, saidas_completas, esperado, obtido
print("Fusões preparadas. ALP e SAM originais preservados.")

In [ ]:
from copy import deepcopy
from pathlib import Path
import pandas as pd
import torch
import torch.nn.functional as F
import models.grid_proto_fewshot as modulo_alp
from models.ProtoSAM import ALPNetWrapper, ALPNetInput

assert VARIANTE == "base"
assert not protosam.training and not protosam.sam.training
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_mask and not protosam.use_neg_points
assert not protosam.coarse_pred_only
assert alpnet.config["proto_grid_size"] == 8
assert "modelos_alp_fusao" not in globals(), (
    "Os modelos ALP das fusões já foram criados."
)

def construir_alp_fusao(encoder_fusao, config):
    class FewShotSegFusao(modulo_alp.FewShotSeg):
        def get_encoder(self):
            self.encoder = encoder_fusao
            self.config["feature_hw"] = [42, 42]

        def get_cls(self):
            assert self.config["cls_name"] == "grid_proto"
            grade = self.config["proto_grid_size"]
            self.cls_unit = modulo_alp.MultiProtoAsConv(
                proto_grid=[grade, grade],
                feature_hw=self.config["feature_hw"],
                embed_dim=encoder_fusao.canais_saida,
            )

        def get_features(self, imgs_concat):
            imgs_concat = F.interpolate(
                imgs_concat,
                size=(672, 672),
                mode="bilinear",
                align_corners=False,
            )
            features = self.encoder(imgs_concat)
            assert tuple(features.shape[1:]) == (
                encoder_fusao.canais_saida, 42, 42
            )
            return features

    modelo = FewShotSegFusao(
        image_size=672,
        pretrained_path=None,
        cfg=deepcopy(config),
    ).to(device=device, dtype=torch.float32)

    modelo.eval().requires_grad_(False)
    assert type(modelo).forward is modulo_alp.FewShotSeg.forward
    return modelo


device = next(encoder.parameters()).device
modelos_alp_fusao = {
    nome: construir_alp_fusao(enc, alpnet.config)
    for nome, enc in encoders_fusao.items()
}
wrappers_fusao = {
    nome: ALPNetWrapper(modelo)
    for nome, modelo in modelos_alp_fusao.items()
}

nome_query_fusao = Path(ds_teste.images[0]).name
assert nome_query_fusao == "cju0u82z3cuma0835wlxrnrjv.png"

item = ds_teste[0]
gt_fusao = item["label"].cpu().numpy() > 0.5
qry_fusao = item["image"].unsqueeze(0).to(device)
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)

def entrada_fusao():
    return ALPNetInput(
        support_images=[sup_img],
        support_labels=[sup_mask],
        query_images=qry_fusao,
        isval=True,
        val_wsize=2,
        show_viz=False,
        supp_fts=None,
    )

def medir_fusao(pred, gt):
    tp = int((pred & gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    return {
        "Dice": 2 * tp / (2 * tp + fp + fn)
                if 2 * tp + fp + fn else 1.0,
        "IoU": tp / (tp + fp + fn) if tp + fp + fn else 1.0,
        "precisao": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "pixels_FG": int(pred.sum()),
    }

mascaras_fusao = {}
linhas_fusao = []
modelo_anterior = protosam.coarse_segmentation_model

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for nome, wrapper in wrappers_fusao.items():
            protosam.coarse_segmentation_model = wrapper

            logits = wrapper(entrada_fusao())
            assert tuple(logits.shape) == (1, 2, 1024, 1024)
            assert torch.isfinite(logits).all().item()
            pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
            del logits

            saida, scores = protosam(
                query_image=qry_fusao,
                coarse_model_input=entrada_fusao(),
                degrees_rotate=0,
            )
            assert tuple(saida.shape) == (1024, 1024)
            assert torch.isfinite(saida).all().item()
            assert ((saida == 0) | (saida == 1)).all().item()
            pred_sam = saida.cpu().numpy() > 0

            for etapa, pred in (("ALP", pred_alp), ("SAM", pred_sam)):
                mascaras_fusao[(nome, etapa)] = pred.copy()
                linhas_fusao.append({
                    "fusao": nome,
                    "etapa": etapa,
                    **medir_fusao(pred, gt_fusao),
                })

            print(f"{nome}: {len(scores)} scores SAM")

finally:
    protosam.coarse_segmentation_model = modelo_anterior

print("Query:", nome_query_fusao)
display(pd.DataFrame(linhas_fusao).round(6))
print("Modelo original reconectado.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import json
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO = Path("/workspace/ProtoSAM-repro")
REF = REPO / "reproducao" / (
    "vmamba_base_k1_estagio2_grade42_20261011T111220763092Z"
)

assert VARIANTE == "base"
assert set(wrappers_fusao) == {"estagios_1_2", "estagios_2_3"}
assert not protosam.training and not protosam.sam.training
assert not protosam.use_cca
assert protosam.point_mode == "both"
assert protosam.use_bbox and protosam.use_points
assert not protosam.use_mask and not protosam.use_neg_points
assert not protosam.coarse_pred_only

for modelo in modelos_alp_fusao.values():
    assert not modelo.training
    assert not any(p.requires_grad for p in modelo.parameters())

referencia = pd.read_csv(REF / "por_imagem.csv")
assert len(referencia) == len(ds_teste) == 100
assert referencia["query"].is_unique
assert referencia["indice"].is_unique

for r in referencia.to_dict("records"):
    assert Path(ds_teste.images[int(r["indice"])]).name == r["query"]

# Guarda as definições executadas no notebook.
nomes_fontes = {"EncoderVMambaFusao", "construir_alp_fusao", "medir_fusao"}
fontes = {}
for fonte in get_ipython().history_manager.input_hist_raw:
    try:
        arvore = ast.parse(fonte)
    except SyntaxError:
        continue
    for no in arvore.body:
        if isinstance(no, (ast.ClassDef, ast.FunctionDef)):
            if no.name in nomes_fontes:
                fontes[no.name] = ast.get_source_segment(fonte, no)

assert set(fontes) == nomes_fontes

def salvar_fusao(path, dados):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(dados, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    tmp.replace(path)

def hash_fusao(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_FUSOES = REPO / "reproducao" / f"vmamba_base_k1_fusoes_{stamp}"
DIR_FUSOES.mkdir(parents=True, exist_ok=False)

for nome in wrappers_fusao:
    for pasta in ("alp", "sam", "registros"):
        (DIR_FUSOES / nome / pasta).mkdir(parents=True)
(DIR_FUSOES / "gt").mkdir()

(DIR_FUSOES / "implementacao.py").write_text(
    "\n\n".join(fontes[n] for n in sorted(fontes)) + "\n",
    encoding="utf-8",
)
referencia.to_csv(DIR_FUSOES / "referencia.csv", index=False)

salvar_fusao(DIR_FUSOES / "config.json", {
    "tipo": "avaliacao exploratoria de fusoes",
    "encoder": "vmamba_base",
    "suporte": SUPORTE_K1,
    "resolucao_encoder": 672,
    "grade": [42, 42],
    "normalizacao": "LayerNorm inicializada por estagio; L2 por posicao",
    "fusao": "concatenacao dividida pela raiz de 2",
    "reducao_espacial": "adaptive_avg_pool2d",
    "ampliacao_espacial": "bilinear, align_corners=False",
    "selecao_sam": "alternativa_0_por_componente",
    "condicoes": {
        nome: {
            "estagios": list(enc.estagios),
            "canais": enc.canais_saida,
            "config_alp": modelos_alp_fusao[nome].config,
        }
        for nome, enc in encoders_fusao.items()
    },
    "referencia": str(REF),
    "gpu_driver": subprocess.check_output([
        "nvidia-smi", "--query-gpu=name,driver_version",
        "--format=csv,noheader",
    ], text=True).strip(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,
    "tf32_cudnn": torch.backends.cudnn.allow_tf32,
    "checkpoint_sha256": hash_fusao(
        REPO / "pretrained_model/vmamba_base_v2.pth"
    ),
    "sam_sha256": hash_fusao(REPO / "pretrained_model/sam_vit_h.pth"),
    "vmamba_sha256": hash_fusao(REPO / "models/vmamba.py"),
    "protosam_sha256": hash_fusao(REPO / "models/ProtoSAM.py"),
})

device = next(encoder.parameters()).device
sup_img = suporte["image"].unsqueeze(0).to(device)
sup_mask = suporte["label"].unsqueeze(0).to(device)
linhas_avaliacao_fusao = []
estado, erro = "executando", None
modelo_anterior = protosam.coarse_segmentation_model

print("Diretório:", DIR_FUSOES, flush=True)

try:
    with torch.inference_mode(), torch.autocast("cuda", enabled=False):
        for r in referencia.sort_values("indice").to_dict("records"):
            nome_query = r["query"]
            item = ds_teste[int(r["indice"])]
            gt = item["label"].cpu().numpy() > 0.5

            with Image.open(REF / "gt" / nome_query) as img:
                gt_ref = np.asarray(img.convert("L")) > 0
            assert np.array_equal(gt, gt_ref), nome_query

            Image.fromarray(gt.astype(np.uint8) * 255).save(
                DIR_FUSOES / "gt" / nome_query
            )
            qry = item["image"].unsqueeze(0).to(device)

            def entrada_avaliacao_fusao():
                return ALPNetInput(
                    support_images=[sup_img],
                    support_labels=[sup_mask],
                    query_images=qry,
                    isval=True, val_wsize=2,
                    show_viz=False, supp_fts=None,
                )

            for nome_fusao, wrapper in wrappers_fusao.items():
                protosam.coarse_segmentation_model = wrapper

                logits = wrapper(entrada_avaliacao_fusao())
                assert tuple(logits.shape) == (1, 2, 1024, 1024)
                assert torch.isfinite(logits).all().item()
                pred_alp = logits.argmax(1)[0].cpu().numpy() > 0
                del logits

                saida, scores = protosam(
                    query_image=qry,
                    coarse_model_input=entrada_avaliacao_fusao(),
                    degrees_rotate=0,
                )
                assert torch.isfinite(saida).all().item()
                assert ((saida == 0) | (saida == 1)).all().item()
                pred_sam = saida.cpu().numpy() > 0

                linha = {
                    "indice": int(r["indice"]),
                    "query": nome_query,
                    "fusao": nome_fusao,
                    "numero_scores_sam": len(scores),
                    "SAM_Dice_estagio2": float(r["SAM_Dice"]),
                }
                for etapa, pred in (("ALP", pred_alp), ("SAM", pred_sam)):
                    assert pred.shape == gt.shape
                    linha.update({
                        f"{etapa}_{k}": v
                        for k, v in medir_fusao(pred, gt).items()
                    })
                    Image.fromarray(pred.astype(np.uint8) * 255).save(
                        DIR_FUSOES / nome_fusao / etapa.lower() / nome_query
                    )

                linha["delta_SAM_vs_estagio2"] = (
                    linha["SAM_Dice"] - linha["SAM_Dice_estagio2"]
                )
                salvar_fusao(
                    DIR_FUSOES / nome_fusao / "registros"
                    / f"{Path(nome_query).stem}.json",
                    linha,
                )
                linhas_avaliacao_fusao.append(linha)

                tmp = DIR_FUSOES / "por_imagem.tmp.csv"
                pd.DataFrame(linhas_avaliacao_fusao).to_csv(tmp, index=False)
                tmp.replace(DIR_FUSOES / "por_imagem.csv")

                print(
                    f"[{len(linhas_avaliacao_fusao)}/200] "
                    f"{nome_fusao} | {nome_query}: "
                    f"ALP={linha['ALP_Dice']:.6f} "
                    f"SAM={linha['SAM_Dice']:.6f}",
                    flush=True,
                )

    estado = "concluida"

except BaseException as exc:
    estado, erro = "interrompida", repr(exc)
    raise

finally:
    protosam.coarse_segmentation_model = modelo_anterior
    salvar_fusao(DIR_FUSOES / "status.json", {
        "status": estado,
        "pares_query_fusao_avaliados": len(linhas_avaliacao_fusao),
        "previstos": 200,
        "erro": erro,
    })
    print("Modelo original reconectado.")

df_fusoes = pd.DataFrame(linhas_avaliacao_fusao)
resumo_fusoes = {}

for nome, grupo in df_fusoes.groupby("fusao"):
    delta = grupo["delta_SAM_vs_estagio2"]
    resumo_fusoes[nome] = {
        "n": len(grupo),
        "medias": {
            etapa: {
                m: float(grupo[f"{etapa}_{m}"].mean())
                for m in ("Dice", "IoU", "precisao", "recall")
            }
            for etapa in ("ALP", "SAM")
        },
        "delta_SAM_medio_vs_estagio2": float(delta.mean()),
        "melhora_vs_estagio2": int((delta > 0).sum()),
        "piora_vs_estagio2": int((delta < 0).sum()),
        "empate_vs_estagio2": int((delta == 0).sum()),
    }

salvar_fusao(DIR_FUSOES / "resumo.json", resumo_fusoes)
print(json.dumps(resumo_fusoes, indent=2))

In [ ]:
from pathlib import Path

treino_dir = Path("/workspace/polyp_raw/TrainDataset/TrainDataset/image")
teste_dir = Path(
    "/workspace/ProtoSAM-repro/data/PolypDataset/TestDataset/test/images"
)
suporte_k1 = "cju3xtufwiv9c0818djsc4cqd.png"

assert treino_dir.is_dir(), treino_dir
assert teste_dir.is_dir(), teste_dir

extensoes = {".png", ".jpg", ".jpeg"}
treino = {
    p.stem for p in treino_dir.iterdir()
    if p.is_file() and p.suffix.lower() in extensoes
}
teste = {
    p.stem for p in teste_dir.iterdir()
    if p.is_file() and p.suffix.lower() in extensoes
}

sobreposicao = sorted(treino & teste)

print("Imagens disponíveis para treino:", len(treino))
print("Imagens no diretório de teste:", len(teste))
print("Nomes presentes nos dois conjuntos:", len(sobreposicao))
print("Primeiras coincidências:", sobreposicao[:20])
print("Suporte K1 presente no treino:", Path(suporte_k1).stem in treino)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import random
import numpy as np
from PIL import Image

REPO = Path("/workspace/ProtoSAM-repro")
TREINO_DIR = Path("/workspace/polyp_raw/TrainDataset/TrainDataset/image")
TESTE_DIR = REPO / "data/PolypDataset/TestDataset/test/images"
SUPORTE_K1 = "cju3xtufwiv9c0818djsc4cqd.png"
SEED = 1234

def listar_imagens(pasta):
    return sorted(
        p for p in pasta.iterdir()
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )

def hash_pixels(caminho):
    with Image.open(caminho) as im:
        pixels = np.asarray(im.convert("RGB"))
    h = hashlib.sha256()
    h.update(str(pixels.shape).encode())
    h.update(pixels.tobytes())
    return h.hexdigest()

originais = listar_imagens(TREINO_DIR)
queries = listar_imagens(TESTE_DIR)
suporte_path = TREINO_DIR / SUPORTE_K1

assert len(originais) == 1450
assert len(queries) == 100
assert suporte_path.is_file()

reservadas = queries + [suporte_path]
nomes_reservados = {p.stem for p in reservadas}
hashes_reservados = {hash_pixels(p) for p in reservadas}

elegiveis, excluidas, duplicadas = [], [], []
hashes_vistos = {}

for p in originais:
    digest = hash_pixels(p)
    registro = {"imagem": str(p.resolve()), "sha256_pixels": digest}

    if p.stem in nomes_reservados or digest in hashes_reservados:
        excluidas.append(registro)
    elif digest in hashes_vistos:
        duplicadas.append({
            **registro,
            "duplicata_de": hashes_vistos[digest],
        })
    else:
        hashes_vistos[digest] = str(p.resolve())
        elegiveis.append(registro)

random.Random(SEED).shuffle(elegiveis)
corte = int(0.9 * len(elegiveis))
treino, validacao = elegiveis[:corte], elegiveis[corte:]

assert treino and validacao
assert not (
    {r["sha256_pixels"] for r in treino}
    & {r["sha256_pixels"] for r in validacao}
)

carimbo = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_DADOS_EFT = REPO / "reproducao" / f"eft_dados_{carimbo}"
DIR_DADOS_EFT.mkdir(parents=True, exist_ok=False)

manifesto = {
    "seed": SEED,
    "suporte_reservado": str(suporte_path.resolve()),
    "queries_reservadas": [str(p.resolve()) for p in queries],
    "treino": treino,
    "validacao": validacao,
    "excluidas_por_reserva": excluidas,
    "duplicadas_removidas": duplicadas,
}

conteudo = json.dumps(manifesto, indent=2, ensure_ascii=False)
(DIR_DADOS_EFT / "manifesto.json").write_text(conteudo, encoding="utf-8")

for nome, registros in [("treino", treino), ("validacao", validacao)]:
    (DIR_DADOS_EFT / f"{nome}.txt").write_text(
        "".join(r["imagem"] + "\n" for r in registros),
        encoding="utf-8",
    )

print("Diretório dos manifestos:", DIR_DADOS_EFT)
print("Excluídas por coincidência com queries/suporte:", len(excluidas))
print("Duplicatas adicionais removidas:", len(duplicadas))
print("Treino:", len(treino))
print("Validação:", len(validacao))
print("SHA256 do manifesto:", hashlib.sha256(conteudo.encode()).hexdigest())

In [ ]:
from pathlib import Path
from PIL import Image
import hashlib

RAIZ = Path("/workspace/polyp_raw")
TREINO = RAIZ / "TrainDataset/TrainDataset/image"
TESTES = RAIZ / "TestDataset/TestDataset"
SUPORTE = "cju3xtufwiv9c0818djsc4cqd.png"

datasets = [
    "Kvasir",
    "CVC-ClinicDB",
    "CVC-ColonDB",
    "ETIS-LaribPolypDB",
]

def hash_pixels(p):
    with Image.open(p) as imagem:
        rgb = imagem.convert("RGB")
        h = hashlib.sha256()
        h.update(str(rgb.size).encode())
        h.update(rgb.tobytes())
        return h.hexdigest()

imagens_treino = sorted(TREINO.glob("*.png"))
assert len(imagens_treino) == 1450

print("Calculando hashes das imagens de treino...")
hashes_treino = {}
for p in imagens_treino:
    hashes_treino.setdefault(hash_pixels(p), []).append(p.name)

nomes_treino = {p.stem for p in imagens_treino}

for dataset in datasets:
    pasta = TESTES / dataset
    imagens = sorted((pasta / "images").glob("*.png"))
    mascaras = sorted((pasta / "masks").glob("*.png"))
    assert imagens, f"Sem imagens: {pasta}"

    nomes_imagens = {p.stem for p in imagens}
    nomes_mascaras = {p.stem for p in mascaras}
    pares_ok = nomes_imagens == nomes_mascaras

    coincidencias = []
    for p in imagens:
        iguais = hashes_treino.get(hash_pixels(p), [])
        if iguais:
            coincidencias.append((p.name, iguais))

    print(f"\n{dataset}: {len(imagens)} imagens, {len(mascaras)} máscaras")
    print("Pares imagem/máscara conferidos por nome:", pares_ok)
    print("Nomes em comum com treino:", len(nomes_imagens & nomes_treino))
    print("Imagens com pixels idênticos ao treino:", len(coincidencias))
    if coincidencias:
        print("Coincidências:", coincidencias)

print("\nSuporte K1 presente no treino:", (TREINO / SUPORTE).is_file())

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

REPO = Path("/workspace/ProtoSAM-repro")
RAIZ = Path("/workspace/polyp_raw")
SUPORTE_K1 = "cju3xtufwiv9c0818djsc4cqd.png"

def pares(pasta_imagens, pasta_mascaras):
    imagens = sorted(pasta_imagens.glob("*.png"))
    registros = []
    for imagem in imagens:
        mascara = pasta_mascaras / imagem.name
        assert mascara.is_file(), f"Máscara ausente: {mascara}"
        registros.append({
            "nome": imagem.name,
            "imagem": str(imagem.resolve()),
            "mascara": str(mascara.resolve()),
        })
    return registros

origem_treino = RAIZ / "TrainDataset/TrainDataset"
pool = pares(origem_treino / "image", origem_treino / "mask")
assert len(pool) == 1450

suportes = [r for r in pool if r["nome"] == SUPORTE_K1]
assert len(suportes) == 1
adaptacao = [r for r in pool if r["nome"] != SUPORTE_K1]
assert len(adaptacao) == 1449

esperados = {
    "Kvasir": 100,
    "CVC-ClinicDB": 62,
    "CVC-ColonDB": 380,
    "ETIS-LaribPolypDB": 196,
}

testes = []
for dataset, quantidade in esperados.items():
    pasta = RAIZ / "TestDataset/TestDataset" / dataset
    registros = pares(pasta / "images", pasta / "masks")
    assert len(registros) == quantidade
    for r in registros:
        testes.append({
            **r,
            "dataset": dataset,
            "id": f"{dataset}/{r['nome']}",
        })

assert len({r["id"] for r in testes}) == 738

carimbo = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
DIR_PROTOCOLO = REPO / "reproducao" / f"protocolo_polipos_{carimbo}"
DIR_PROTOCOLO.mkdir(parents=True, exist_ok=False)

manifesto = {
    "descricao": "Inventário para EFT e avaliação em quatro datasets",
    "suporte_k1": suportes[0],
    "pool_adaptacao": adaptacao,
    "teste": testes,
    "divisao_treino_validacao": "Ainda não definida",
    "observacao": (
        "Máscaras manuais do pool registradas para rastreabilidade; "
        "não devem ser usadas como alvos no EFT autossupervisionado."
    ),
}

arquivo = DIR_PROTOCOLO / "manifesto.json"
arquivo.write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Manifesto:", arquivo)
print("Pool de adaptação:", len(adaptacao))
print("Suporte reservado:", SUPORTE_K1)
print("Queries de teste:", len(testes))